# A Practical Guide to Quantitative Finance Interviews — Chapter 2 Brain Teasers & Chapter 3 Calculus

**Xinfeng Zhou ("the Green Book")** — worked solutions in code.

This notebook grows as I read through the book. Each brain teaser gets:

* a short **restatement** of the problem,
* a **markdown explanation of the logic**, and
* **one general-purpose function** that solves *every* instance (all $n$, arbitrary
  parameters) — not just the specific numbers in the book.

Covered so far:

| § | Theme | Problems |
|---|-------|----------|
| **2.1** | Problem Simplification | Screwy pirates · Tiger and sheep |
| **2.2** | Logic Reasoning | River crossing · Birthday problem · Card game · Burning ropes · Defective ball · Trailing zeros · Horse race · Infinite power tower |
| **2.3** | Thinking Out of the Box | Box packing · Calendar cubes · Door to offer · Message delivery · Last ball · Light switches · Quant salary |
| **2.4** | Application of Symmetry | Coin piles · Mislabeled bags · Wise men |
| **2.5** | Series Summation | Clock pieces · Missing integers · Counterfeit coins · Glass balls |
| **2.6** | The Pigeon Hole Principle | Matching socks · Handshakes · Have we met before? · Ants on a square · Counterfeit coins II |
| **2.7** | Modular Arithmetic | Prisoner problem · Division by 9 · Chameleon colors |
| **2.8** | Math Induction | Coin split · Chocolate bar · Race track |
| **2.9** | Proof by Contradiction | Rainbow hats |

The recurring meta-technique of §2.1 is: *when a problem is too big, solve the
smallest case, then grow it.* In code that is just **dynamic programming /
recursion** — build the answer for size $n$ from the answer for size $n-1$.
§2.3 is deliberately lighter on code: several of its puzzles turn on a single
*aha* with no algorithm to speak of, so those are written up as prose.

**Chapter 3** begins after §2.9 — calculus **reference notes** (differentiation and integration: rules, standard tables, and applications), written as markdown because that material is formula-centric rather than code.

---
## 2.1 Problem Simplification

> *If the original problem is too complex to solve at once, identify a simplified
> version and start there — the simplest sub-problem — and gradually add
> complexity until a pattern emerges.*

### 2.1.1 Screwy pirates

**Problem.** $N$ pirates (ranked by seniority) split $M$ gold coins. The most
senior proposes a split; **all** present pirates vote; if **at least half**
approve, it passes, otherwise the proposer is thrown overboard and the next
senior proposes. Every pirate is perfectly rational with priorities:
**(1) survive, (2) maximise gold, (3) all else equal, prefer fewer pirates left.**
How are the coins divided?

**Logic.** The 5-pirate case is hopeless head-on, so *simplify*:

* **1 pirate:** takes all $M$ coins.
* **2 pirates:** the proposer's own vote is already 50% → he keeps everything,
  junior gets 0.
* **3 pirates:** if the proposer dies we fall back to the 2-pirate outcome where
  pirate 1 gets 0. So pirate 1 accepts **1 coin** (strictly better than 0). Two
  votes → passes.
* **$p$ pirates:** the proposer needs $\lceil p/2\rceil$ votes (his own plus
  $\lceil p/2\rceil-1$ bribes). A junior sells his vote for **one more coin than
  he'd get if the proposer died** — i.e. than his payoff in the already-solved
  $(p-1)$-pirate game. So bribe the *cheapest* juniors (those who'd get 0, or who'd
  *die* in the sub-game and vote yes just to live).

That is a **bottom-up DP**: `dp[p]` is built from `dp[p-1]`. It also captures the
famous coin-limited regime — with only 100 coins a very senior proposer may be
unable to buy enough votes and gets thrown overboard.

In [1]:
from math import ceil

def screwy_pirates(n_pirates, coins):
    """Rational pirate split. Pirates 1..N by seniority (N = first proposer).
    Returns payoff[1..N] (index 0 unused); a pirate thrown overboard is 'dead'.
    Bottom-up DP: the p-pirate outcome is built from the (p-1)-pirate sub-game."""
    dp = {1: {1: coins}}                      # lone pirate takes everything
    for p in range(2, n_pirates + 1):
        prev = dp[p - 1]                      # outcome if proposer p is thrown over
        bribes_needed = ceil(p / 2) - 1       # 'at least 50%' minus his own vote
        # price of each junior's vote: dead-in-sub-game -> 0 (wants to live),
        # else one coin more than his sub-game payoff
        cost = {j: (0 if prev.get(j, 'dead') == 'dead' else prev[j] + 1)
                for j in range(1, p)}
        cheapest = sorted(range(1, p), key=cost.get)[:bribes_needed]
        bill = sum(cost[j] for j in cheapest)
        if bill <= coins:                     # proposer can buy a majority
            dp[p] = {p: coins - bill,
                     **{j: (cost[j] if j in cheapest else 0) for j in range(1, p)}}
        else:                                 # can't -> thrown overboard
            dp[p] = {**prev, p: 'dead'}
    final = dp[n_pirates]
    return [None] + [final.get(i, 'dead') for i in range(1, n_pirates + 1)]

# Book case: 5 pirates, 100 coins  ->  [1, 0, 1, 0, 98]
print("5 pirates, 100 coins:", screwy_pirates(5, 100)[1:])
for n in (2, 3, 4, 5, 6, 10):
    print(f"{n:>2} pirates, 100 coins:", screwy_pirates(n, 100)[1:])
# Coin-limited regime: proposer #203 cannot buy enough votes and dies
r = screwy_pirates(205, 100)
print("205 pirates: proposer #205 ->", r[205], "| survivors =",
      sum(x != 'dead' for x in r[1:]))

5 pirates, 100 coins: [1, 0, 1, 0, 98]
 2 pirates, 100 coins: [0, 100]
 3 pirates, 100 coins: [1, 0, 99]
 4 pirates, 100 coins: [0, 1, 0, 99]
 5 pirates, 100 coins: [1, 0, 1, 0, 98]
 6 pirates, 100 coins: [0, 1, 0, 1, 0, 98]
10 pirates, 100 coins: [0, 1, 0, 1, 0, 1, 0, 1, 0, 96]
205 pirates: proposer #205 -> dead | survivors = 204


### 2.1.2 Tiger and sheep

**Problem.** $n$ perfectly-rational tigers and 1 sheep are on an island. A tiger
that eats the sheep *becomes* a sheep (and is then edible by the others). Tigers
prefer to eat, but survival comes first. Is the sheep eaten?

**Logic.** Simplify by induction on $n$:

* $n=1$: the lone tiger eats — nothing can punish it.
* $n=2$: eating turns you into a sheep facing 1 hungry tiger → you'd be eaten. So
  neither eats. Sheep **safe**.
* In general a tiger eats **iff** the resulting $(n-1)$-tiger world is *safe* for
  its new sheep-self. Hence `eaten(n) = not eaten(n-1)`, with `eaten(1)=True`.

The pattern: the sheep is eaten **iff $n$ is odd**.

In [2]:
from functools import lru_cache

@lru_cache(None)
def sheep_eaten(n_tigers):
    """True iff a lone sheep is eaten by n rational tigers.
    eaten(n) = not eaten(n-1); eaten(1) = True  ->  odd n means eaten."""
    if n_tigers <= 0:
        return False
    return not sheep_eaten(n_tigers - 1)

for n in [1, 2, 3, 4, 5, 99, 100]:
    print(f"{n:>3} tigers -> sheep {'EATEN' if sheep_eaten(n) else 'safe'}")

  1 tigers -> sheep EATEN
  2 tigers -> sheep safe
  3 tigers -> sheep EATEN
  4 tigers -> sheep safe
  5 tigers -> sheep EATEN
 99 tigers -> sheep EATEN
100 tigers -> sheep safe


---
## 2.2 Logic Reasoning

> *Carefully chain what each fact/observation implies, discarding impossibilities
> until only the answer remains.*

### 2.2.1 River crossing (bridge & torch)

**Problem.** People must cross a bridge that holds **2 at a time**; a single torch
must travel on every crossing, so a pair moves at the **slower** person's pace.
Minimise total time. (Book: times $10,5,2,1 \Rightarrow 17$ min.)

**Logic.** Sort times ascending. The bottleneck is the two **slowest**; the trick
is to ferry them across *together* so their times overlap, using the fastest
people as shuttles. Getting the two slowest ($y \le z$) across, with the two
fastest ($a \le b$) available, costs the cheaper of two schedules:

* **A — send the fast pair as ferries:** $\;2b + a + z$
  ($a,b$ over; $a$ back; $y,z$ over; $b$ back).
* **B — fastest shuttles each slow one:** $\;2a + y + z$.

Peel off the two slowest, add the cheaper cost, repeat until $\le 3$ remain (whose
optima are immediate). This greedy-DP is optimal for **any** number of people and
**any** times.

In [3]:
def bridge_and_torch(times):
    """Minimum time to move everyone across a 2-person bridge with one torch.
    General for any number of people and any crossing times.
    Returns (total_time, moves)."""
    t = sorted(times); n = len(t)
    if n <= 2:
        return (t[-1] if t else 0,
                [("cross", tuple(t))] if t else [])
    moves, total, hi = [], 0, n - 1
    while hi >= 3:
        a, b, y, z = t[0], t[1], t[hi - 1], t[hi]      # 2 fastest, 2 slowest
        if 2 * b + a + z <= 2 * a + y + z:             # A: fast pair ferries
            moves += [("cross", (a, b)), ("back", (a,)),
                      ("cross", (y, z)), ("back", (b,))]
            total += 2 * b + a + z
        else:                                          # B: fastest shuttles
            moves += [("cross", (a, z)), ("back", (a,)),
                      ("cross", (a, y)), ("back", (a,))]
            total += 2 * a + y + z
        hi -= 2
    if hi == 2:                                        # last three
        moves += [("cross", (t[0], t[1])), ("back", (t[0],)), ("cross", (t[0], t[2]))]
        total += t[1] + t[0] + t[2]
    else:                                              # last two
        moves += [("cross", (t[0], t[1]))]; total += t[1]
    return total, moves

total, moves = bridge_and_torch([10, 5, 2, 1])
print("Book case [10,5,2,1] -> total =", total, "min")
for m in moves:
    print("  ", m)
print("General [1,2,5,10,15,20] ->", bridge_and_torch([1, 2, 5, 10, 15, 20])[0], "min")

Book case [10,5,2,1] -> total = 17 min
   ('cross', (1, 2))
   ('back', (1,))
   ('cross', (5, 10))
   ('back', (2,))
   ('cross', (1, 2))
General [1,2,5,10,15,20] -> 42 min


### 2.2.2 Birthday problem

**Problem.** Boss $A$'s birthday is one of 10 dates. **You** are told only the
**month**, colleague **C** only the **day**. Then:

1. *You:* "I don't know $A$'s birthday, **and I know C doesn't either**."
2. *C:* "Now I know it."
3. *You:* "Now I know it too."

What is the birthday? (Green Book dates: Mar 4/5/8, Jun 4/7, Sep 1/5, Dec 1/2/8.)

**Logic** — each statement is a *filter* on the candidate set:

1. You are certain **C** can't know ⇒ your month contains **no globally-unique
   day** ⇒ eliminate every month that holds a day appearing only once (that kills
   the months of `Jun 7` and `Dec 2`, i.e. June and December).
2. **C** now knows ⇒ among survivors his **day is unique** ⇒ keep dates whose day
   is unique (kills the day shared by March and September).
3. **You** now know ⇒ among survivors your **month is unique**.

One date survives. This engine works for **any** date set + this statement
pattern.

In [4]:
from collections import Counter

def birthday_deduction(dates):
    """Solve the 'boss's birthday' common-knowledge puzzle for any date set.
    dates: list of (month, day). Applies the three public statements as filters."""
    uniq = lambda pool, idx: {v for v, k in Counter(p[idx] for p in pool).items() if k == 1}
    S = list(dates)
    bad_months = {m for (m, d) in S if d in uniq(S, 1)}    # (1) you're sure C can't know
    S = [x for x in S if x[0] not in bad_months]
    S = [x for x in S if x[1] in uniq(S, 1)]               # (2) C now knows -> day unique
    S = [x for x in S if x[0] in uniq(S, 0)]               # (3) you now know -> month unique
    return S

green_book = [("Mar", 4), ("Mar", 5), ("Mar", 8), ("Jun", 4), ("Jun", 7),
              ("Sep", 1), ("Sep", 5), ("Dec", 1), ("Dec", 2), ("Dec", 8)]
print("A's birthday is:", birthday_deduction(green_book))

A's birthday is: [('Sep', 1)]


### 2.2.3 Card game

**Problem.** A deck has $R$ red and $B$ black cards (book: $26$ each). Cards are
turned two at a time: **red-red** → your pile, **black-black** → dealer's pile,
**mixed** → discarded. You win \$100 if your pile is **strictly larger**. What is
the game worth?

**Logic (invariant / symmetry).** Every *mixed* discard removes exactly one red
and one black. So each red card ends up either in your pile or in a mixed discard,
and likewise each black. Counting:

$$\text{(your pile)} - \text{(dealer pile)} = R - B \quad\text{— always, for every shuffle.}$$

With $R=B$ the two piles are **always equal** → you can never have *more* → the
game is worth **\$0**. (More generally you're guaranteed to win iff $R>B$.)

In [5]:
def card_game_value(reds, blacks, payoff=100):
    """Value of the red/black pairing game. Invariant: your_pile - dealer_pile
    == reds - blacks for every arrangement, so the outcome is guaranteed."""
    diff = reds - blacks
    if diff > 0:
        return payoff, "You always win."
    if diff < 0:
        return 0, "You can never win -> pay nothing."
    return 0, "Always a tie -> you never have MORE -> pay nothing."

print("26 red / 26 black:", card_game_value(26, 26))
print("28 red / 24 black:", card_game_value(28, 24))

26 red / 26 black: (0, 'Always a tie -> you never have MORE -> pay nothing.')
28 red / 24 black: (100, 'You always win.')


### 2.2.4 Burning ropes

**Problem.** Two ropes each burn for **60 min** but *unevenly* (you can't trust
any fraction of a rope to take a proportional time). Measure **45 min**.

**Logic.** Lighting a rope at **both ends** always finishes in $T/2$ *regardless*
of the uneven density (the two flames jointly consume the whole rope). So:

* Light rope A at **both** ends and rope B at **one** end.
* When A is gone (**30 min**), B has exactly 30 min of burn left — now light B's
  **other** end, halving it → **15 min** more. Total **45 min**.

Generalisation: repeatedly halving lets you measure any $60\cdot(\text{dyadic
rational})$; the function lists the reachable durations for $k$ ropes.

In [6]:
from fractions import Fraction

def burning_ropes_45():
    """Constructive schedule to measure 45 min with two 60-min ropes."""
    return ["t=0 : light rope A at BOTH ends, rope B at ONE end",
            "t=30: A burned out -> light B's OTHER end (B has 30 min left -> halves to 15)",
            "t=45: B burned out. Total = 45 minutes"]

def rope_measurable_times(n_ropes, unit=60, halvings=3):
    """Durations reachable with n identical `unit`-minute ropes by lighting ends:
    unit * (dyadic rationals) up to n*unit."""
    vals = {Fraction(w) + Fraction(k, 2 ** m)
            for w in range(n_ropes + 1) for m in range(halvings + 1)
            for k in range(2 ** m + 1)}
    return sorted(v * unit for v in vals if 0 < v <= n_ropes)

for line in burning_ropes_45():
    print(line)
print("Reachable with 2 ropes (min):", [round(float(v), 2) for v in rope_measurable_times(2)])

t=0 : light rope A at BOTH ends, rope B at ONE end
t=30: A burned out -> light B's OTHER end (B has 30 min left -> halves to 15)
t=45: B burned out. Total = 45 minutes
Reachable with 2 ropes (min): [7.5, 15.0, 22.5, 30.0, 37.5, 45.0, 52.5, 60.0, 67.5, 75.0, 82.5, 90.0, 97.5, 105.0, 112.5, 120.0]


### 2.2.5 Defective ball

**Problem.** Among $n$ identical balls exactly one is defective — **heavier *or*
lighter**, you don't know which. Using a balance (tells which pan is heavier, or
balance), find the defective ball **and** whether it's heavy/light. (Book:
$n=12$ in 3 weighings.)

**Logic (information counting → ternary codes).** Each weighing has 3 outcomes
(left down / balance / right down), so $k$ weighings distinguish $3^k$ cases. There
are $2n$ possibilities ($n$ balls × heavy/light); we also must reject the "no
weighing moved" contradiction and keep heavy/light distinguishable, giving the
classic bound

$$\frac{3^{k}-3}{2} \ge n .$$

**Construction (one plan for all $n$).** Give each ball a distinct code in
$\{L,R,N\}^k$: in weighing $c$ put its $L$-balls left, $R$-balls right. Choose codes
so that (i) no code and its $L\leftrightarrow R$ **mirror** are both used (so a
heavy ball and its mirror light ball never collide) and (ii) every column has
equal $L$s and $R$s (pans balance). The observed outcome string then **is** the
defective ball's code (heavy) or its mirror (light). $n=12,k=3$ falls straight out.

In [7]:
def min_weighings(n, direction_known=False):
    """Fewest balance weighings for one defective among n balls.
    unknown heavy/light: (3^k-3)/2 >= n;  known direction: 3^k >= n."""
    k = 0
    while (3 ** k if direction_known else (3 ** k - 3) // 2) < n:
        k += 1
    return k

_mirror = lambda code: tuple('R' if c == 'L' else 'L' if c == 'R' else 'N' for c in code)

def build_weighing_plan(n, k=None):
    """General non-adaptive plan: assign each of n balls a {L,R,N}^k code with no
    used mirror pair and balanced pans. Returns (codes, weighings)."""
    from itertools import product
    if k is None:
        k = min_weighings(n)
    pairs, seen = [], set()
    for c in product('LRN', repeat=k):                 # canonical rep per mirror pair
        if all(x == 'N' for x in c) or c in seen:
            continue
        seen.add(c); seen.add(_mirror(c))
        pairs.append(c if next(x for x in c if x != 'N') == 'L' else _mirror(c))
    vec = lambda code: [1 if x == 'L' else -1 if x == 'R' else 0 for x in code]
    chosen = []
    def bt(i, run, need):                              # pick n pairs, sign them -> pans balance
        if need == 0:
            return all(v == 0 for v in run)
        if i >= len(pairs) or len(pairs) - i < need:
            return False
        v = vec(pairs[i])
        for s in (1, -1):
            chosen.append(pairs[i] if s == 1 else _mirror(pairs[i]))
            if bt(i + 1, [run[c] + s * v[c] for c in range(k)], need - 1):
                return True
            chosen.pop()
        return bt(i + 1, run, need)                    # or skip this pair
    if not bt(0, [0] * k, n):
        raise ValueError(f"n={n} needs more than k={k} weighings")
    codes = list(chosen)
    weighings = [([b for b in range(n) if codes[b][c] == 'L'],
                  [b for b in range(n) if codes[b][c] == 'R']) for c in range(k)]
    return codes, weighings

def find_defective(n, defective, heavier, plan=None):
    """Identify the defective ball and its type from the plan's outcomes."""
    codes, weighings = plan or build_weighing_plan(n)
    obs = []
    for left, right in weighings:
        d = 1 if heavier else -1
        w = (d if defective in left else 0) - (d if defective in right else 0)
        obs.append('L' if w > 0 else 'R' if w < 0 else 'N')
    obs = tuple(obs)
    for b, code in enumerate(codes):
        if obs == code:            return b, True     # its own side sank -> heavy
        if obs == _mirror(code):   return b, False    # its own side rose  -> light
    raise RuntimeError("undecodable")

print("min weighings: n=12 ->", min_weighings(12), "| n=100 ->", min_weighings(100))
plan = build_weighing_plan(12)
print("Weighings for 12 balls (left pan vs right pan):")
for i, (L, R) in enumerate(plan[1], 1):
    print(f"  W{i}: {L} vs {R}")
# verify every one of the 24 hidden scenarios is solved correctly
ok = all(find_defective(12, b, h, plan) == (b, h) for b in range(12) for h in (True, False))
print("All 24 hidden (ball, heavy/light) scenarios identified in 3 weighings:", ok)

min weighings: n=12 -> 3 | n=100 -> 5
Weighings for 12 balls (left pan vs right pan):
  W1: [0, 1, 2, 3] vs [4, 5, 6, 7]
  W2: [0, 1, 2, 4] vs [3, 8, 9, 10]
  W3: [0, 3, 6, 9] vs [1, 5, 8, 11]
All 24 hidden (ball, heavy/light) scenarios identified in 3 weighings: True


### 2.2.6 Trailing zeros

**Problem.** How many trailing zeros does $100!$ have?

**Logic.** A trailing zero is a factor of $10 = 2\times5$. In $n!$ factors of $2$
vastly outnumber factors of $5$, so **count the 5s** (Legendre's formula):

$$Z_5(n) = \left\lfloor \tfrac n5\right\rfloor + \left\lfloor \tfrac n{25}\right\rfloor + \left\lfloor \tfrac n{125}\right\rfloor + \cdots$$

For $n=100$: $20 + 4 = 24$. Generalised to any base $b$: factor $b=\prod p^{e}$,
count each prime's exponent in $n!$, divide by $e$, take the minimum.

In [8]:
def _factorize(m):
    f, d = {}, 2
    while d * d <= m:
        while m % d == 0:
            f[d] = f.get(d, 0) + 1; m //= d
        d += 1
    if m > 1:
        f[m] = f.get(m, 0) + 1
    return f

def trailing_zeros_factorial(n, base=10):
    """Trailing zeros of n! in the given base (Legendre's formula)."""
    zeros = None
    for p, e in _factorize(base).items():
        cnt, pk = 0, p
        while pk <= n:
            cnt += n // pk; pk *= p
        zeros = cnt // e if zeros is None else min(zeros, cnt // e)
    return zeros or 0

print("trailing zeros of 100! =", trailing_zeros_factorial(100))
print("trailing zeros of 1000! =", trailing_zeros_factorial(1000))
print("trailing zeros of 100! in base 12 =", trailing_zeros_factorial(100, 12))

trailing zeros of 100! = 24
trailing zeros of 1000! = 249
trailing zeros of 100! in base 12 = 48


### 2.2.7 Horse race

**Problem.** $25$ horses, a track with $5$ lanes, no timer. Fewest races to find
the **3 fastest**? (Answer: **7**.)

**Logic.**

1. **5 group races** (horses 1-5, …, 21-25) rank each group of 5.
2. **1 race of the 5 group winners** → the overall **fastest**, and it prunes whole
   groups: only a group whose winner placed 1st/2nd/3rd can contribute more
   top-3 horses.
3. After that, exactly **5 horses** remain eligible for places 2 and 3 (the
   triangular pruning leaves $\binom{m+1}{2}-1$ contenders) → **1 final race**.

Total $5+1+1 = 7$. The function returns the count for general $n$, lanes, and
top-$m$ (exact for the usual $m \le \text{lanes}$).

In [9]:
import math

def horse_races(n_horses, lanes, top_m):
    """Races (standard method) to find the fastest top_m of n horses, `lanes`
    at a time, no clock. Returns (races, explanation)."""
    if top_m > n_horses:
        raise ValueError("top_m > n_horses")
    groups = math.ceil(n_horses / lanes)
    races = groups + 1                                   # group races + winners' race
    if top_m == 1:
        return races, "winner of the winners' race is fastest"
    contenders = top_m * (top_m + 1) // 2 - 1            # eligible for places 2..top_m
    extra = math.ceil(contenders / lanes)
    return races + extra, f"{groups} group + 1 winners' + {extra} runoff"

print("25 horses, 5 lanes, top 3 ->", horse_races(25, 5, 3))
print("49 horses, 7 lanes, top 3 ->", horse_races(49, 7, 3))

25 horses, 5 lanes, top 3 -> (7, "5 group + 1 winners' + 1 runoff")
49 horses, 7 lanes, top 3 -> (9, "7 group + 1 winners' + 1 runoff")


### 2.2.8 Infinite sequence (power tower)

**Problem.** Solve $x^{x^{x^{\cdot^{\cdot^{\cdot}}}}} = 2$.

**Logic.** Let $y$ be the whole (infinite) tower. Because it is infinite, the
exponent *is itself* $y$, so $x^{y}=y$. With $y=2$: $x^{2}=2 \Rightarrow x=\sqrt2$.

General: $x^{y}=y \Rightarrow x = y^{1/y}$. **Caveat:** the tower only *converges*
for $x\in[e^{-e},\,e^{1/e}]$, i.e. targets $y\in(0,e]$. So $y=2$ gives
$x=\sqrt2$ (valid), while "solving" $y=4$ also gives $x=\sqrt2$ — but that tower
actually converges to $2$, not $4$ (the classic trap). The function returns $x$
**and** a convergence check.

In [10]:
import math

def tower_base_for(target):
    """Solve x^(x^(x^...)) = target -> x = target**(1/target).
    Returns (x, converges, actual_limit). Tower converges only for target in (0, e]."""
    x = target ** (1.0 / target)
    converges = math.exp(-math.e) <= x <= math.exp(1.0 / math.e)
    v = 1.0
    for _ in range(1000):
        v = x ** v
    return x, converges, v

for target in (2.0, 4.0):
    x, conv, lim = tower_base_for(target)
    note = "OK" if abs(lim - target) < 1e-6 else f"TRAP: tower actually -> {lim:.2f}"
    print(f"target={target}: x = target**(1/target) = {x:.6f}  ({note})")

target=2.0: x = target**(1/target) = 1.414214  (OK)
target=4.0: x = target**(1/target) = 1.414214  (TRAP: tower actually -> 2.00)


---
## 2.3 Thinking Out of the Box

> *When the obvious framing admits no solution, change the representation:
> add a dimension, exploit a physical property, or use a degree of freedom the
> naive reading ignored.*

This section leans on insight rather than computation, so the pure *aha* puzzles
below are written up as prose; only the three with genuine algorithmic content
(Box packing, Last ball, Quant salary) carry code.

### 2.3.1 Box packing

**Problem.** Can you pack **53** bricks of size $1\times1\times4$ into a
$6\times6\times6$ box? (Volume is $53\cdot4 = 212 < 216$, so volume alone doesn't
forbid it.)

**Logic (out-of-the-box colouring).** Volume counting is a dead end — you need a
*colouring* invariant. Cut the cube into $27$ blocks of $2\times2\times2$ and
3-D-checkerboard-colour them: $14$ blocks of one colour, $13$ of the other. A
$1\times1\times4$ brick, however placed, occupies exactly $2$ unit cells in a
black $2\times2\times2$ block and $2$ in a white one. Each $2\times2\times2$
block has only $8$ cells → at most $4$ brick-halves. The **minority** colour
($13$ blocks) can therefore host at most $13\times4 = 52$ brick-halves of its
colour ⇒ **at most 52 bricks**. So $53$ is impossible. The function returns this
colouring bound for any even cube.

In [11]:
def packing_coloring_bound(n):
    """Upper bound on the number of 1x1x4 bricks that fit in an n x n x n box
    (n even), via the 2x2x2-block 3-D checkerboard colouring argument.
    Returns (coloring_bound, naive_volume_bound)."""
    assert n % 2 == 0, "argument needs an even side"
    s = n // 2                                   # blocks per axis
    black = sum((x + y + z) % 2 == 0
                for x in range(s) for y in range(s) for z in range(s))
    white = s ** 3 - black
    coloring_bound = 4 * min(black, white)       # <=4 brick-halves per minority block
    return coloring_bound, n ** 3 // 4

cb, vb = packing_coloring_bound(6)
print(f"6x6x6 box: colouring bound = {cb} bricks (volume alone allows {vb})")
print(f"Requested 53 bricks -> {'possible' if 53 <= cb else 'IMPOSSIBLE'} (53 > {cb})")

6x6x6 box: colouring bound = 52 bricks (volume alone allows 54)
Requested 53 bricks -> IMPOSSIBLE (53 > 52)


### 2.3.2 Calendar cubes

**Problem.** Put single-digit numbers on the six faces of **two** dice so their
front faces can show every day of the month, **01–31** (single days as `01`–`09`,
and you may swap the two dice).

**Answer (aha).** Both dice must carry **0, 1, 2**: `11` and `22` force a `1` and a
`2` on *each* die, and displaying `01`–`09` forces a `0` on each. That uses 3
faces per die and leaves **6 faces for the digits 3, 4, 5, 6, 7, 8, 9 — seven
digits, six faces.** The trick: a **`6` can be read upside-down as a `9`**, since
they are never needed simultaneously. So

* **Die A:** 0, 1, 2, 3, 4, 5
* **Die B:** 0, 1, 2, 6, 7, 8   (the `6` doubles as `9`)

covers `01` through `31`. Pure representational insight — nothing to compute.

### 2.3.3 Door to offer

**Problem.** Two doors (offer / exit), each with a guard; one guard **always
lies**, the other **always tells the truth**, and you don't know which is which.
With a **single yes/no question to one guard**, find the offer door.

**Answer (aha).** Ask *either* guard: **"Would the *other* guard say your door
leads to the offer?"** Whatever the truth, one guard reports it faithfully and the
other inverts it, so the two-guard chain always yields **one inversion** of
reality. Hence the answer is *always reversed*: if he says **"yes," take the other
door; if "no," take this door.** The self-referential phrasing cancels the unknown
of who lies — no computation required.

### 2.3.4 Message delivery

**Problem.** Send a document to a colleague through an insecure messenger:
anything in an **unlocked** box is lost, and each of you owns a padlock whose
**only key you keep yourself**. How do you get the document there securely?

**Answer (aha).** You can't share a key, but a box can hold **more than one lock**.

1. You lock the box with **your** padlock and send it.
2. Your colleague adds **their** padlock (two locks now) and sends it back.
3. You remove **your** lock and send it again.
4. Your colleague removes **their** lock and opens the box.

The box is locked at every leg of every trip, yet no key ever travels. (This is
exactly the intuition behind Diffie–Hellman-style key exchange.)

### 2.3.5 Last ball

**Problem.** A bag has **20 blue** and **14 red** balls. Repeatedly remove two at
random: if **same colour**, add back a **blue**; if **different**, add back a
**red**. What colour is the **last** ball? And for **20 blue, 13 red**?

**Logic (parity invariant).** Track how red count $R$ changes per step:

| drawn | net change |
|-------|------------|
| blue, blue | $R \to R$ |
| red, red   | $R \to R-2$ (remove 2 red, add 1 blue) |
| red, blue  | $R \to R$ (remove 1 red +1 blue, add 1 red) |

$R$ only ever stays equal or drops by $2$ — its **parity never changes**. The bag
shrinks by one each step until one ball remains, so the last ball is **red iff the
initial red count is odd**. Hence $14 \Rightarrow$ blue, $13 \Rightarrow$ red.

In [12]:
def last_ball(blue, red):
    """Colour of the final ball under the same->blue / different->red rule.
    Red count parity is invariant, so the answer depends only on red % 2."""
    if blue + red == 0:
        return None
    return "red" if red % 2 == 1 else "blue"

print("20 blue, 14 red ->", last_ball(20, 14))
print("20 blue, 13 red ->", last_ball(20, 13))
print("general 7 blue, 100 red ->", last_ball(7, 100))

20 blue, 14 red -> blue
20 blue, 13 red -> red
general 7 blue, 100 red -> blue


### 2.3.6 Light switches

**Problem.** One bulb in a room, **four** switches outside (all off, exactly one
controls the bulb). You may toggle freely but can enter the room **only once**.
Identify the controlling switch.

**Answer (aha).** On/off is only **1 bit** — enough for 2 switches. The extra
channel is that a bulb **gets hot** when lit. Using *(state × temperature)* gives
$2\times2 = 4$ distinguishable outcomes:

1. Turn on switches **1 and 2**; wait several minutes.
2. Turn **2 off**, turn **3 on**, and immediately enter.

| bulb | verdict |
|------|---------|
| on & hot  | switch 1 |
| off & hot | switch 2 |
| on & cold | switch 3 |
| off & cold| switch 4 |

The insight is finding a **second physical variable** (heat), not any algorithm.

### 2.3.7 Quant salary

**Problem.** Eight quants want the **average** of their salaries but nobody will
reveal their own figure. Compute the average without anyone learning another's
salary.

**Logic (masked running sum / secure aggregation).** The first quant adds a
**secret random offset** $r$ to their salary and passes the total on; each
subsequent quant adds their own salary and passes it along; the total returns to
the first quant, who **subtracts $r$** and divides by $n$. Every value in transit
is masked by $r$ (and by the unknown salaries), so no individual figure leaks,
yet the final sum is exact. The demo below confirms it recovers the true average
for any group.

In [13]:
import random

def average_salary_protocol(salaries, seed=0):
    """Privacy-preserving average via a masked running sum. Each party only ever
    sees (secret offset + partial sum of others), never an individual salary.
    Returns the computed average, identical to the true average."""
    rng = random.Random(seed)
    offset = rng.randint(10 ** 6, 10 ** 9)       # party 1's secret mask
    running = offset
    transcript = []                              # what each party passes on (all masked)
    for s in salaries:
        running += s
        transcript.append(running)
    total = running - offset                     # party 1 removes the mask
    return total / len(salaries), transcript

avg, transcript = average_salary_protocol([120, 145, 160, 135, 200, 175, 150, 165])
print("computed average salary =", avg, "(k)")
print("true average            =", sum([120,145,160,135,200,175,150,165]) / 8, "(k)")
print("values passed between quants (all masked, no salary is recoverable):")
print("  ", transcript)

computed average salary = 156.25 (k)
true average            = 156.25 (k)
values passed between quants (all masked, no salary is recoverable):
   [907691179, 907691324, 907691484, 907691619, 907691819, 907691994, 907692144, 907692309]


---
## 2.4 Application of Symmetry

> *Look for a transformation that leaves the essential structure unchanged — a
> flip, a pairing, a relabelling. The quantity that survives the symmetry is
> usually the answer, and it often holds no matter what you cannot see.*

Two of these turn on a physical or logical symmetry with a tidy algorithm (coin
piles, wise men); the mislabeled-bags puzzle is a pure deduction, so it is written
up as prose.

### 2.4.1 Coin piles

**Problem.** $n$ coins lie on a table; exactly $m$ show **heads**, the rest tails.
You are **blindfolded** and cannot tell a head from a tail. Split the coins into
**two piles** that contain the **same number of heads**. (Book: $n=1000$, $m=20$.)

**Logic (symmetry of a flip).** You cannot *find* the heads, but you can *balance*
them. Move any $m$ coins into pile $A$ and **flip every coin in $A$**; leave the
other $n-m$ as pile $B$. Say your blind grab happened to scoop up $h$ of the $m$
heads. Then

* pile $A$ holds $h$ heads and $m-h$ tails → after flipping, $m-h$ **heads**;
* pile $B$ holds the remaining $m-h$ **heads**.

The unknown $h$ **cancels** — the two counts are equal for *every* arrangement, so
the strategy works without ever seeing a coin. (If $m=0$ both piles trivially hold
$0$ heads.)

In [14]:
import random

def coin_split(n_coins, n_heads, arrangement=None):
    """Blindfolded coin problem. n_coins lie on a table with n_heads showing
    heads; you cannot tell heads from tails. Split them into two piles holding the
    SAME number of heads.

    Strategy (works for EVERY hidden arrangement): move any n_heads coins into
    pile A and flip them all; pile B is the untouched rest. If pile A scooped up h
    of the heads it then shows (n_heads - h) heads after flipping -- exactly what
    pile B keeps. The unknown h cancels, so equality holds blindfolded.

    Returns (heads_in_A, heads_in_B) -- always equal. Random layout if none given;
    1 = heads, 0 = tails.
    """
    if not 0 <= n_heads <= n_coins:
        raise ValueError("need 0 <= n_heads <= n_coins")
    if arrangement is None:
        arrangement = [1] * n_heads + [0] * (n_coins - n_heads)
        random.shuffle(arrangement)
    if len(arrangement) != n_coins or sum(arrangement) != n_heads:
        raise ValueError("arrangement inconsistent with (n_coins, n_heads)")
    pileA = [1 - c for c in arrangement[:n_heads]]     # grab n_heads coins, flip all
    pileB = arrangement[n_heads:]                       # leave the rest untouched
    return sum(pileA), sum(pileB)

random.seed(0)
for n, m in [(1000, 20), (100, 20), (52, 13), (5, 5), (7, 0)]:
    results = [coin_split(n, m) for _ in range(500)]   # 500 random hidden layouts
    always_equal = all(a == b for a, b in results)
    seen = sorted({a for a, _ in results})             # the (common) head count per layout
    print(f"{n:>4} coins, {m:>2} heads: take {m}, flip -> piles ALWAYS match "
          f"({always_equal}); per-layout head count ranged over {seen}")

1000 coins, 20 heads: take 20, flip -> piles ALWAYS match (True); per-layout head count ranged over [17, 18, 19, 20]


 100 coins, 20 heads: take 20, flip -> piles ALWAYS match (True); per-layout head count ranged over [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


  52 coins, 13 heads: take 13, flip -> piles ALWAYS match (True); per-layout head count ranged over [5, 6, 7, 8, 9, 10, 11, 12, 13]
   5 coins,  5 heads: take 5, flip -> piles ALWAYS match (True); per-layout head count ranged over [0]
   7 coins,  0 heads: take 0, flip -> piles ALWAYS match (True); per-layout head count ranged over [0]


### 2.4.2 Mislabeled bags

**Problem.** Three bags hold, respectively, **apples**, **oranges**, and a **mix**
of both. Each bag carries a label — `apple`, `orange`, or `mix` — but **every
label is wrong**. Drawing fruit one at a time (without looking inside), what is the
**fewest draws** that lets you relabel all three correctly?

**Answer (aha).** **One** draw — from the bag labelled `mix`. The trick is which
symmetry the "all wrong" constraint breaks: the bag marked `mix` cannot be mixed,
so it is **pure** apples *or* pure oranges, and a single fruit settles which.

* Draw an **apple** ⇒ the `mix` bag is really **apples**.
* The bag labelled `orange` cannot be oranges (its label is wrong) and cannot be
  apples (already found) ⇒ it must be the **mix**.
* The remaining bag, labelled `apple`, is therefore **oranges**.

(Drawing an orange is the mirror image.) Sampling any *other* bag can leave you
stuck: a fruit from the bag labelled `apple` — which is orange or mix — might come
up orange and fail to separate the two. The bag labelled `mix` is special because
its wrong label rules out **two** contents at once (not-mix leaves only the two
pure options), so one draw fully resolves it. That asymmetry *is* the puzzle — no
computation required.

### 2.4.3 Wise men

**Problem.** A sultan holds $n$ wise men (book: $50$). In a room sits a single
glass, standing **bottom-down**. Every minute the sultan summons **one wise man at
random** (any man, any number of times, forever); the summoned man may **turn the
glass over or leave it**, and sees the glass only while he is inside. At any point
a man may announce **"every one of us has now been summoned at least once."** A
correct announcement frees them all; a wrong one is fatal. Beforehand they may
agree on a strategy but afterwards never communicate except through the glass.
Find a strategy that **guarantees** freedom.

**Logic (a symmetric one-bit channel + a counter).** The glass is a single bit
that anyone can read or write. The men are interchangeable, so **break the
symmetry** by appointing **one counter**; everyone else is identical ("ordinary").

* **Counter:** whenever the glass is **up**, turn it back **down** and add $1$ to a
  private tally. Announce the moment the tally reaches $n-1$.
* **Ordinary man:** the **first and only** time you ever find the glass **down**,
  turn it **up**; on every other visit, do nothing.

The glass starts **down**, and each of the $n-1$ ordinary men raises it **exactly
once** in his life. Every raise is eventually seen and reset by the counter, who
therefore records exactly one per *distinct* ordinary man. When the tally hits
$n-1$, all $n-1$ ordinary men have been in the room — and so, obviously, has the
counter — so the announcement is **never** wrong. Correctness needs only that
every man is summoned eventually, which random selection guarantees; the *expected*
wait grows like $n^2$.

In [15]:
import random

def wise_men_glass(n, seed=None, max_minutes=50_000_000):
    """Green Book 'Wise men'. n wise men; one glass starts bottom-DOWN. Each minute
    a uniformly-random man is summoned and may flip the glass; men see it only while
    inside. Anyone may declare 'all n have now been summoned at least once' -- right
    frees everyone, wrong is fatal.

    Winning strategy: appoint one COUNTER, the other n-1 are ordinary.
      * COUNTER: on finding the glass UP, turn it DOWN and tally +1; declare at n-1.
      * ORDINARY man: the first and only time he finds it DOWN, he turns it UP.
    The glass starts down and each ordinary man raises it exactly once, so the
    counter's tally counts distinct men; tally == n-1 => everyone has been in.
    Returns the minute of the (always-correct) declaration.
    """
    if n < 1:
        raise ValueError("need n >= 1")
    if n == 1:
        return 1                                        # lone man: first summons suffices
    rng = random.Random(seed)
    COUNTER, glass_up, tally = 0, False, 0              # glass_up False = bottom-down
    raised = [False] * n                                # has this ordinary man raised it?
    called = [False] * n                                # ground truth, only to verify
    for minute in range(1, max_minutes + 1):
        m = rng.randrange(n)
        called[m] = True
        if m == COUNTER:
            if glass_up:                                # a raise is waiting: count & reset
                glass_up, tally = False, tally + 1
                if tally == n - 1:
                    assert all(called)                  # provably everyone was summoned
                    return minute
        elif not glass_up and not raised[m]:            # ordinary man's one-time raise
            glass_up, raised[m] = True, True
    raise RuntimeError("exceeded max_minutes")

for n in (2, 3, 10, 50):
    runs = [wise_men_glass(n, seed=s) for s in range(20)]
    print(f"n={n:>3}: counter declares at tally n-1={n-1:>2}; minutes over 20 runs -> "
          f"min {min(runs):>5}, avg {sum(runs)//len(runs):>5}, max {max(runs):>5}")
print("strategy ALWAYS frees them; expected wait grows ~ n^2")

n=  2: counter declares at tally n-1= 1; minutes over 20 runs -> min     2, avg     4, max    10
n=  3: counter declares at tally n-1= 2; minutes over 20 runs -> min     5, avg     9, max    17
n= 10: counter declares at tally n-1= 9; minutes over 20 runs -> min    73, avg   123, max   210
n= 50: counter declares at tally n-1=49; minutes over 20 runs -> min  2028, avg  2617, max  3088
strategy ALWAYS frees them; expected wait grows ~ n^2


---
## 2.5 Series Summation

> *Spot a total that a closed-form sum pins down exactly — $1+2+\cdots+n=\frac{n(n+1)}{2}$,
> $\sum i^2=\frac{n(n+1)(2n+1)}{6}$, or the triangular reach $\frac{k(k+1)}{2}$ — and
> the puzzle collapses to arithmetic on that formula.*

All four carry code. Clock pieces is the one fixed instance (the numbers $1$–$12$
are locked in place both in value and position), so it is not generalised to $n$;
the other three are solved for arbitrary size.

### 2.5.1 Clock pieces

**Problem.** A clock face (numbers **1–12**) falls and breaks into **three
pieces**. Remarkably, the numbers on each piece add up to the **same total**.
Which numbers lie on each piece?

**Logic (series summation).** The whole face sums to
$$1+2+\cdots+12=\frac{12\cdot 13}{2}=78,$$
so three equal pieces must each total $78/3=\mathbf{26}$ — and equal pieces exist
at all only because $3$ **divides** $78$. Hitting $26$ forces each piece to **pair
large numbers with small ones** (a top-heavy arc like $10,11,12$ already
overshoots), which gives the book's grouping

$$\{1,2,11,12\},\qquad\{3,4,9,10\},\qquad\{5,6,7,8\}\qquad(\text{each }=26).$$

The same divisor argument says *how many* equal pieces are possible: only the
divisors of $78$ a 12-number face can realise — $2$ (39 each), $3$ (26), or $6$
(13, the six diametrically-opposite pairs). Because the $12$ labels are **fixed**
both in value and position, this puzzle is **not** generalised to $n$; the solver
below keeps the clock's numbers and finds an equal-sum split (several exist — the
book's balanced one is shown above).

In [16]:
def clock_partition(pieces=3):
    """Break a clock face (numbers 1..12, FIXED) into `pieces` groups whose numbers
    sum to the same total. Series-summation core: 1+2+...+12 = 12*13/2 = 78, so each
    group must sum to 78/pieces -- possible only when `pieces` divides 78. When the
    12 numbers also split evenly, groups are kept the same size, reproducing the
    book's balanced 'pair a big number with a small one' answer. The 12 labels are
    fixed, so this puzzle is NOT generalised to n.
    Returns (partition, total, target); partition is None when impossible."""
    nums = list(range(12, 0, -1))                       # largest-first for a fast search
    total = sum(nums)                                   # 78
    if total % pieces:
        return None, total, None
    target = total // pieces
    cap = len(nums) // pieces if len(nums) % pieces == 0 else len(nums)   # equal sizes
    groups, sums = [[] for _ in range(pieces)], [0] * pieces
    def bt(i):
        if i == len(nums):
            return True
        seen = set()
        for g in range(pieces):
            if sums[g] in seen or len(groups[g]) >= cap:   # skip twins / full groups
                continue
            seen.add(sums[g])
            if sums[g] + nums[i] <= target:
                groups[g].append(nums[i]); sums[g] += nums[i]
                if bt(i + 1):
                    return True
                groups[g].pop(); sums[g] -= nums[i]
        return False
    ok = bt(0)
    return (sorted(sorted(g) for g in groups) if ok else None), total, target

print("clock face sums to 1+...+12 =", sum(range(1, 13)))
print("book's 3-piece answer {1,2,11,12},{3,4,9,10},{5,6,7,8} -> sums",
      1 + 2 + 11 + 12, 3 + 4 + 9 + 10, 5 + 6 + 7 + 8)
for p in (2, 3, 6, 5):
    part, tot, tgt = clock_partition(p)
    if part is None and tgt is None:
        print(f"  {p} pieces: impossible ({p} does not divide {tot})")
    else:
        print(f"  {p} pieces, each summing {tgt}: {part}")

clock face sums to 1+...+12 = 78
book's 3-piece answer {1,2,11,12},{3,4,9,10},{5,6,7,8} -> sums 26 26 26
  2 pieces, each summing 39: [[1, 2, 3, 10, 11, 12], [4, 5, 6, 7, 8, 9]]
  3 pieces, each summing 26: [[1, 3, 10, 12], [2, 4, 9, 11], [5, 6, 7, 8]]
  6 pieces, each summing 13: [[1, 12], [2, 11], [3, 10], [4, 9], [5, 8], [6, 7]]
  5 pieces: impossible (5 does not divide 78)


### 2.5.2 Missing integers

**Problem.** The integers $1,2,\dots,n$ are handed to you in some order but **one
or two are missing**. Recover the missing value(s) in a single streaming pass,
using only a couple of running totals — no length-$n$ lookup table.

**Logic (power sums).** The complete set has totals fixed by closed forms:

$$S_1=\sum_{i=1}^{n} i=\frac{n(n+1)}{2},\qquad
  S_2=\sum_{i=1}^{n} i^2=\frac{n(n+1)(2n+1)}{6}.$$

* **One missing** $m$: subtract what you saw — $m = S_1 - \sum\text{present}$.
* **Two missing** $a,b$: the shortfalls give $a+b = S_1 - \sum\text{present}$ and
  $a^2+b^2 = S_2 - \sum\text{present}^2$; then
  $ab = \tfrac{(a+b)^2-(a^2+b^2)}{2}$, and $a,b$ are the roots of
  $t^2-(a+b)\,t+ab$.

Two unknowns, two equations — solved for any $n$ with $O(1)$ extra memory.

In [17]:
from math import isqrt

def find_missing(n, present):
    """Numbers missing from `present` (a collection drawn from 1..n) via series
    summation, in O(1) extra memory. Handles 1 or 2 missing values.
      1 missing:  m = Sum(1..n) - Sum(present),          Sum i  = n(n+1)/2
      2 missing:  a+b and a^2+b^2 give two equations,    Sum i^2 = n(n+1)(2n+1)/6
    """
    S1 = n * (n + 1) // 2
    S2 = n * (n + 1) * (2 * n + 1) // 6
    d1 = S1 - sum(present)                              # a + b   (or the lone missing)
    d2 = S2 - sum(x * x for x in present)               # a^2 + b^2
    k = n - len(present)
    if k == 0:
        return []
    if k == 1:
        return [d1]
    if k == 2:
        ab = (d1 * d1 - d2) // 2                        # a*b from (a+b)^2 - (a^2+b^2)
        r = isqrt(d1 * d1 - 4 * ab)                     # (a - b)
        return [(d1 - r) // 2, (d1 + r) // 2]
    raise ValueError("this power-sum method handles at most 2 missing")

import random
random.seed(1)
for n, miss in [(100, [42]), (100, [17, 83]), (1000, [500, 501]), (9, [1, 9])]:
    present = [x for x in range(1, n + 1) if x not in miss]
    random.shuffle(present)                             # order does not matter
    print(f"n={n:>4}, removed {miss} -> recovered {sorted(find_missing(n, present))}")

n= 100, removed [42] -> recovered [42]
n= 100, removed [17, 83] -> recovered [17, 83]
n=1000, removed [500, 501] -> recovered [500, 501]
n=   9, removed [1, 9] -> recovered [1, 9]


### 2.5.3 Counterfeit coins

**Problem.** $n$ bags (book: $10$) each hold identical coins that should weigh
$10$ g. **One** bag is counterfeit — every coin in it weighs $9$ g **or** $11$ g
(you know the magnitude of the error, not which bag). With a **digital scale**
that gives an exact reading, find the bad bag in **one** weighing.

**Logic (series summation as a ruler).** Take a *different* number of coins from
each bag: $1$ from bag $1$, $2$ from bag $2$, …, $n$ from bag $n$ —
$\frac{n(n+1)}{2}$ coins in all, expected to weigh $10\cdot\frac{n(n+1)}{2}$ g.
Only bag $i$'s coins are off, and you took exactly $i$ of them, so the scale
deviates by $(\pm 1)\cdot i$. The **magnitude** of the miss names the bag
($|{\rm deviation}| = i$) and its **sign** tells light ($9$ g) from heavy
($11$ g). Distinct coin counts turn a single number into $n$ separable signals —
solved for any $n$.

In [18]:
def find_counterfeit_bag(n, fake_bag, fake_coin_weight, real_weight=10):
    """One-weighing solution for n bags, one of which is counterfeit. Take i coins
    from bag i (i=1..n); on a digital scale the total deviates from the expected
    real_weight * Sum(1..n) by exactly (fake - real) * fake_bag, so |deviation|
    names the bag and its sign says light/heavy. Sum(1..n) = n(n+1)/2 is the
    series-summation trick that keeps every bag's signature distinct."""
    expected = real_weight * (n * (n + 1) // 2)         # 10 * Sum(1..n)
    actual = expected + (fake_coin_weight - real_weight) * fake_bag
    dev = actual - expected
    bag = abs(dev) // abs(fake_coin_weight - real_weight)
    return bag, ("heavy" if dev > 0 else "light")

for n in (10, 25):
    ok = all(find_counterfeit_bag(n, b, w) == (b, "heavy" if w > 10 else "light")
             for b in range(1, n + 1) for w in (9, 11))
    print(f"n={n} bags: one weighing pinpoints the bag AND light/heavy in every case: {ok}")
print("e.g. 10 bags, bag 7 counterfeit at 9g ->", find_counterfeit_bag(10, 7, 9))

n=10 bags: one weighing pinpoints the bag AND light/heavy in every case: True
n=25 bags: one weighing pinpoints the bag AND light/heavy in every case: True
e.g. 10 bags, bag 7 counterfeit at 9g -> (7, 'light')


### 2.5.4 Glass balls

**Problem.** You have **two** identical glass balls and a **100-storey** building.
There is a critical floor at or above which a dropped ball shatters. Find it while
**minimising the worst-case number of drops** (a broken ball is gone for good).

**Logic (triangular reach).** With two balls the first is a *coarse* probe: once it
breaks at floor $f$, the second must climb one floor at a time from the last safe
level. To cap the worst case at $k$ drops, drop the first ball at floor $k$, then
$k+(k-1)$, then $k+(k-1)+(k-2)$, … — each survival spends one drop but the
remaining budget shrinks by one, so after $k$ drops the plan has covered

$$k+(k-1)+\cdots+1=\frac{k(k+1)}{2}\ \text{floors}.$$

The fewest drops is thus the least $k$ with $\frac{k(k+1)}{2}\ge\text{floors}$; for
$100$ that is $k=\mathbf{14}$ (since $\tfrac{13\cdot14}{2}=91<100\le105=\tfrac{14\cdot15}{2}$).
For $e$ balls the reach after $k$ drops generalises to $\sum_{i=1}^{e}\binom{k}{i}$,
which the function below inverts for any building height.

In [19]:
from math import comb

def min_drops(floors, balls=2):
    """Fewest worst-case drops to find the critical floor among `floors` with
    `balls` identical glass balls. Two-ball case is pure series summation: k drops
    reach k+(k-1)+...+1 = k(k+1)/2 floors, so take the least k with k(k+1)/2 >=
    floors (100 -> 14). General e-ball reach after k drops is sum_{i=1..e} C(k,i)."""
    reach = lambda k: sum(comb(k, i) for i in range(1, balls + 1))
    k = 0
    while reach(k) < floors:
        k += 1
    return k

def two_ball_schedule(floors):
    """First-ball drop floors for the optimal 2-ball plan (each gap shrinks by 1)."""
    k = min_drops(floors, 2)
    f, step, plan = 0, k, []
    while f < floors and step > 0:
        f += step
        plan.append(min(f, floors))
        step -= 1
    return plan

for fl in (100, 200, 1000):
    print(f"{fl:>4} floors, 2 balls -> {min_drops(fl, 2):>2} worst-case drops")
print("100 floors, 2 balls -> drop first ball at floors:", two_ball_schedule(100))
print("100 floors:  1 ball ->", min_drops(100, 1), "(linear scan) | 3 balls ->", min_drops(100, 3))

 100 floors, 2 balls -> 14 worst-case drops
 200 floors, 2 balls -> 20 worst-case drops
1000 floors, 2 balls -> 45 worst-case drops
100 floors, 2 balls -> drop first ball at floors: [14, 27, 39, 50, 60, 69, 77, 84, 90, 95, 99, 100]
100 floors:  1 ball -> 100 (linear scan) | 3 balls -> 9


---
## 2.6 The Pigeon Hole Principle

> *If more than $n$ objects are placed into $n$ boxes, then at least one box holds
> two or more of them. Choose the boxes cleverly and a "must exist" claim falls out
> with no construction at all.*

**The principle.**

* **Basic form.** Put $n+1$ pigeons into $n$ holes and some hole gets $\ge 2$
  pigeons. Nothing says *which* hole — only that a collision is unavoidable.
* **Generalised form.** Put $N$ objects into $k$ boxes and some box holds at least
  $\left\lceil N/k \right\rceil$ objects. (The basic form is just $N=n+1$, $k=n$,
  where $\lceil (n+1)/n\rceil = 2$.)

**Why it is powerful.** It is a pure *existence* tool: it proves something must
happen without ever exhibiting it — exactly what "can you **guarantee** …?"
interview questions demand. Every problem below is the same two-step move:

1. **Pick the pigeons and the holes.** The whole difficulty is deciding what counts
   as an object and what counts as a box — a colour, a possible handshake-count, a
   relationship, a patch of floor.
2. **Compare the counts.** Once there are more pigeons than holes (or more than
   $k\,(t-1)$ of them), a box with $\ge t$ objects is forced.

This section is all reasoning and no code — the pigeon hole principle yields
*proofs of inevitability*, not algorithms to run.

### 2.6.1 Matching socks

**Problem.** A drawer holds socks of a few different colours — in the book's
instance **2 red, 20 yellow, and 31 blue**. The room is dark, so you cannot see
colours. How many socks must you pull out to be **certain** of holding a matching
pair?

**Pigeon holes = colours.** Make one box per colour ($3$ boxes here). Each sock you
draw is a pigeon dropped into the box of its colour, and a "matching pair" is simply
*two pigeons in the same box*.

* Draw $3$ socks and you might be unlucky — one red, one yellow, one blue — one per
  box, still no pair.
* Draw a **4th** sock and, with only $3$ colour-boxes, it is *forced* to join a box
  that already holds one. That collision is your pair.

So with $c$ colours the answer is $\mathbf{c+1}$ (here $3+1 = 4$) — one more than
the number of holes. Notice the *quantities* $2, 20, 31$ are irrelevant to
guaranteeing a generic pair; only the **number of colours** (holes) matters. They
would come into play only for a different question — say, guaranteeing a pair of one
*specific* colour, whose worst case is drawing every other colour first.

### 2.6.2 Handshakes

**Problem.** You arrive at a welcome party with **25** team members — **26** people
in all. Everyone shakes some hands (nobody shakes their own). Can you be *certain*
that **two people shook exactly the same number of hands**?

**Pigeon holes = possible handshake counts.** Among $n$ people each person shook
somewhere between $0$ and $n-1$ hands — that looks like $n$ possible values for $n$
people, not yet a squeeze. The insight is that the two **extremes cannot coexist**:

* if **someone shook $n-1$ hands** they shook with *everyone*, so **nobody** can sit
  at $0$;
* if **someone shook $0$ hands** they shook with no one, so **nobody** can sit at
  $n-1$.

So at most $n-1$ of the values $\{0,1,\dots,n-1\}$ are actually occupied — that is
$n-1$ holes for $n$ pigeons. By the pigeon hole principle **two people share a
handshake count**. With $n = 26$: $26$ people, at most $25$ available counts, so
**yes** — and the argument holds for any party of $n \ge 2$.

### 2.6.3 Have we met before?

**Problem.** Six people are at a party. Any two of them either **have met before**
(mutual acquaintances) or **have not** (mutual strangers). Show that there must be a
group of **three who all know each other**, *or* a group of **three who are all
mutual strangers**.

**Pigeon holes = the two relationship types.** Fix one person, call her $A$. She has
a relationship with each of the other **five** people, and each is one of **two
kinds** — "met" or "not met." Five relationships into two boxes: by the generalised
pigeon hole principle one box holds at least $\lceil 5/2 \rceil = \mathbf{3}$ of
them. Say $A$ has **met** at least three others; call three of them $B, C, D$ (the
all-strangers case is identical with the words swapped).

Now look only among $B, C, D$:

* If **any** pair of them — say $B$ and $C$ — have **also met**, then $A, B, C$ are
  three mutual acquaintances. Done.
* If **no** pair among $B, C, D$ has met, then $B, C, D$ are themselves three mutual
  strangers. Done.

Either way the trio exists. This is the classic **friends-and-strangers theorem**
(the Ramsey number $R(3,3) = 6$): one pigeon hole step forces three same-type
relationships out of a single person, and a short case check finishes it. Six is the
smallest party for which the claim is unavoidable — with five people it can fail.

### 2.6.4 Ants on a square

**Problem.** **51 ants** sit on a square tile of side length **1**. You hold a
circular glass of radius $\tfrac{1}{7}$. Can you always place the glass (flat on the
tile) so that it covers **at least 3 ants** — whatever the arrangement?

**Pigeon holes = a grid of small squares.** Cut the unit square into a $5\times5$
grid of **25** smaller squares, each of side $\tfrac{1}{5}$. The $51$ ants are the
pigeons and the $25$ cells are the holes, so by the generalised pigeon hole
principle some cell contains at least
$$\left\lceil \frac{51}{25} \right\rceil = \lceil 2.04 \rceil = \mathbf{3}\ \text{ants}.$$

**Why the glass fits.** A $\tfrac{1}{5}\times\tfrac{1}{5}$ square is spanned by the
circle through its corners, of radius half the diagonal,
$$r = \tfrac{1}{2}\cdot\frac{\sqrt2}{5} = \frac{\sqrt2}{10} \approx 0.1414,$$
and $\tfrac{\sqrt2}{10} < \tfrac{1}{7} \approx 0.1429$. So a glass of radius
$\tfrac17$ centred on that cell **completely covers** it — and hence the $\ge 3$
ants inside it. The two ideas mesh perfectly: pigeon hole guarantees a crowded cell,
and the radius $\tfrac17$ is *just* big enough to swallow a whole cell. The design is
tight — with only $50$ ants a cell might hold just $2$, which is why the count is
$51 = 2\cdot 25 + 1$.

### 2.6.5 Counterfeit coins II

**Problem.** There are **5 bags**, each holding 100 coins. Every coin weighs
**9, 10, or 11 grams**; within a bag all coins are identical, but you do **not**
know which of the three weights a bag holds. Using a **digital scale exactly
once**, identify the coin type in **every** bag.

**Why a naive weighing fails (pigeon hole).** Encode each bag by its deviation from
$10$ g: $d_i \in \{-1, 0, +1\}$, so the five bags together form one of
$3^5 = 243$ possible type-assignments — these are the **pigeons**. A single weighing
returns just one number, whose only useful content is the total deviation
$\sum_i c_i d_i$, where $c_i$ is how many coins you drew from bag $i$; the distinct
readings it can produce are the **holes**. If you draw too few coins the reading
spans fewer than $243$ values, so by the pigeon hole principle two different
assignments land on the same weight and become indistinguishable. Concretely, one
coin from each of bags $1$ and $2$ gives deviation $d_1 + d_2 \in \{-2,\dots,2\}$ —
only $5$ possible sums for the $3\times 3 = 9$ combinations, so collisions are
forced.

**Fix: give every assignment its own hole (balanced ternary).** Draw a different,
base-$3$-spaced number of coins from each bag —
$$1\ \text{from bag 1},\ 3\ \text{from bag 2},\ 9\ \text{from bag 3},\ 27\ \text{from bag 4},\ 81\ \text{from bag 5}.$$
Now the deviation is $\sum_{i=1}^{5} 3^{\,i-1} d_i$ with $d_i \in \{-1,0,1\}$ —
exactly a number written in **balanced ternary**. Each of the $243$ assignments maps
to a *distinct* integer in $[-121, 121]$ (and those $243$ integers fill the range
exactly), so #holes $=$ #pigeons and the map is injective: the single reading
decodes uniquely back to $(d_1,\dots,d_5)$. The expected weight if every coin were
$10$ g is $10\cdot(1+3+9+27+81) = 1210$ g; the signed gap from $1210$, read off in
balanced ternary, spells out each bag's type.

This is **Counterfeit coins I** (§2.5.3) pushed up a level. There a *single* bag was
off by a known $\pm 1$, so plain counts $1,2,\dots,n$ gave each bag a separate
signature; here **every** bag carries information and has **three** possible states,
so the counts must be spaced by **powers of $3$** to keep all $3^n$ assignments in
separate holes. The pigeon hole principle is what tells you the spacing has to be
this generous — anything tighter guarantees a collision.

---
## 2.7 Modular Arithmetic

> *Two integers are congruent mod $m$, written $x \equiv y \pmod m$, when they leave
> the same remainder on division by $m$ — equivalently when $m \mid (x-y)$. Working
> "mod $m$" collapses the infinitely many integers into just $m$ residue classes
> $\{0,1,\dots,m-1\}$, and any quantity a problem forces to stay in one class becomes
> an invariant.*

**The facts we lean on.** Congruence mod $m$ is preserved by addition, subtraction,
and multiplication: if $a \equiv a'$ and $b \equiv b' \pmod m$ then

$$a+b \equiv a'+b',\qquad a-b \equiv a'-b',\qquad ab \equiv a'b' \pmod m.$$

So you may reduce mod $m$ at *any* stage of a calculation without changing the final
residue — which is exactly what makes a remainder a robust invariant. Two
consequences drive the problems below:

* **Place values collapse.** If $b \equiv 1 \pmod m$ then $b^{\,i} \equiv 1$ for every
  $i$, so a base-$b$ number is congruent to the **sum of its digits** mod $m$.
* **A conserved residue is a wall.** If every legal move leaves some combination of
  the state unchanged mod $m$, then any target with a *different* residue is
  **unreachable** — ruled out with no construction at all.

The three problems are one such idea each: a shared sum mod $k$ broadcast by one
prisoner, a digit-sum congruence mod $9$, and a conserved difference mod $3$. Where a
problem has a free size parameter it is solved for general $n$; Division by 9 is a
pure proof and is written up in markdown only.

### 2.7.1 Prisoner problem

**Problem.** $n$ prisoners (book: $100$) are lined up front-to-back and each is given
a hat of one of $k$ colours (book: $k=2$, red/blue). Every prisoner sees the hats of
**all those in front** of him but not his own and not those behind. Starting from the
**back**, each prisoner in turn calls out a single colour — a guess of his own hat —
and everyone hears every guess. They may agree on a strategy in advance but cannot
otherwise communicate. How many can be **guaranteed** correct?

**Label the colours $0,1,\dots,k-1$ and count mod $k$.** The snag is that the back
prisoner sees everyone else yet has *no* information about his own hat, so he cannot
be saved for certain. Spend him as a **broadcast**: let him announce

$$S \;=\; \Big(\textstyle\sum_{j=0}^{n-2} h_j\Big) \bmod k,$$

the sum of every hat he can see, reduced mod $k$. Now all the others share one public
number $S$ that, by construction, satisfies $S \equiv \sum_{j=0}^{n-2} h_j \pmod k$ —
a congruence involving *every* hat except the back one.

**Peel off one unknown at a time.** Take prisoner $i$ (for $0 \le i \le n-2$). By the
time it is his turn he has

* **heard** the true colours of everyone between him and the broadcaster, i.e.
  $h_{i+1},\dots,h_{n-2}$ (each was announced correctly — see below), and
* **seen** the hats ahead of him, $h_0,\dots,h_{i-1}$.

Every term of $S \equiv \sum_{j=0}^{n-2} h_j$ is therefore known to him **except his
own** $h_i$. Because subtraction is well defined mod $k$, he isolates it:

$$h_i \;\equiv\; S \;-\; \underbrace{\sum_{j=0}^{i-1} h_j}_{\text{seen ahead}} \;-\; \underbrace{\sum_{j=i+1}^{n-2} h_j}_{\text{heard behind}} \pmod k .$$

Since $h_i \in \{0,\dots,k-1\}$ there is exactly **one** residue that fits — his hat
is pinned uniquely. He announces it (correctly), which hands the next prisoner one
more known term, and the deduction cascades all the way to the front.

**Guarantee.** Prisoners $0$ through $n-2$ — that is $\mathbf{n-1}$ of them — are
**always** correct. The back prisoner is right only when his own hat happens to equal
$S$, probability $1/k$, and with no information about it he can do no better. (For the
book's $k=2$: $99$ of $100$ are saved for certain, and the broadcast is just the
**parity** — the even/odd count of red hats.)

In [20]:
def prisoner_hats(hats, k):
    """Prisoners 0..n-1 in a line wear hats coloured 0..k-1; prisoner n-1 is at the
    BACK and sees everyone ahead, prisoner 0 at the front sees no one. Guessing from
    the back forward, each says one colour and all hear it. Strategy: the back
    prisoner announces S = (sum of the hats he sees) mod k; every other prisoner
    subtracts the hats he SEES ahead and the true hats he has HEARD behind him from
    S, leaving his own hat as the only unknown in the congruence. O(n) via a prefix
    sum. Returns the guesses; all but possibly the back prisoner are certain."""
    n = len(hats)
    guess = [None] * n
    S = sum(hats[:-1]) % k                       # back prisoner encodes the sum mod k
    guess[-1] = S                                # his own hat is unknown to him
    pref = [0] * (n + 1)                          # pref[j] = sum(hats[:j])
    for j in range(n):
        pref[j + 1] = pref[j] + hats[j]
    heard = 0                                     # running sum of hats announced behind i
    for i in range(n - 2, -1, -1):
        guess[i] = (S - pref[i] - heard) % k     # pref[i] = hats he sees ahead
        heard += guess[i]
    return guess

import random
random.seed(0)
for n, k in [(100, 2), (100, 3), (10, 5)]:
    T = 500
    front_ok = back_ok = 0
    for _ in range(T):
        hats = [random.randrange(k) for _ in range(n)]
        g = prisoner_hats(hats, k)
        front_ok += all(g[i] == hats[i] for i in range(n - 1))   # front n-1 must be exact
        back_ok += (g[-1] == hats[-1])                            # back is a 1/k gamble
    print(f"n={n:>3}, k={k}: front {n-1} all correct in {front_ok}/{T} trials; "
          f"back prisoner correct {back_ok}/{T}  (~1/k = {1/k:.2f})")

n=100, k=2: front 99 all correct in 500/500 trials; back prisoner correct 244/500  (~1/k = 0.50)
n=100, k=3: front 99 all correct in 500/500 trials; back prisoner correct 185/500  (~1/k = 0.33)
n= 10, k=5: front 9 all correct in 500/500 trials; back prisoner correct 112/500  (~1/k = 0.20)


### 2.7.2 Division by 9

**Problem.** Give a rule to decide whether an arbitrary positive integer is divisible
by $9$, and **prove** it.

**Rule.** Add up the decimal digits; the number is divisible by $9$ **iff** that digit
sum is. (Repeat on the digit sum if you like — you land on a single digit, the
*digital root*, and $9 \mid N$ exactly when the digital root is $9$.)

**Proof (base-10 place values collapse mod 9).** Write the number by its digits
$d_m d_{m-1}\cdots d_1 d_0$, meaning

$$N \;=\; \sum_{i=0}^{m} d_i \, 10^{\,i}.$$

The whole engine is the single congruence $10 \equiv 1 \pmod 9$ (because $10-1 = 9$).
Since multiplication respects congruence, raising both sides to the $i$-th power gives

$$10^{\,i} \equiv 1^{\,i} = 1 \pmod 9 \qquad \text{for every } i \ge 0.$$

Multiply by the digit $d_i$ and sum over $i$ — addition respects congruence too — so

$$N \;=\; \sum_{i=0}^{m} d_i\,10^{\,i} \;\equiv\; \sum_{i=0}^{m} d_i \cdot 1 \;=\; \sum_{i=0}^{m} d_i \pmod 9 .$$

Thus $N$ and its digit sum have the **same remainder mod 9**; in particular one is a
multiple of $9$ exactly when the other is. Each iteration replaces a number by a
strictly smaller one of the same residue until a single digit remains, so the rule
terminates at the digital root. $\;\blacksquare$

**What else falls out for free.** The proof used only $10 \equiv 1$, so it holds
verbatim for every divisor of $9$ — the **rule for 3 is identical** (since $3 \mid 9$).
It also generalises to any base $b$: as $b \equiv 1 \pmod{b-1}$, a base-$b$ numeral is
congruent to its digit sum mod $b-1$ (e.g. mod $15$ in hexadecimal). The sibling fact
$b \equiv -1 \pmod{b+1}$ gives the *alternating* digit-sum test — mod $11$ in base 10.

### 2.7.3 Chameleon colors

**Problem.** An island has $13$ red, $15$ green, and $17$ blue chameleons. Whenever
two chameleons of **different** colours meet, both switch to the **third** colour
(e.g. a red and a green meeting leaves two blues). Can the colony ever become a
**single** colour?

**Encode a move.** Track the counts $(r,g,b)$. A meeting of the two colours other than
$Z$ decrements each of them and adds $2$ to $Z$ — for instance red+green $\to$ blue
sends $(r,g,b)\mapsto(r-1,\,g-1,\,b+2)$. The total $r+g+b$ never changes
($-1-1+2 = 0$).

**Find the invariant (work mod 3).** Reduce the pairwise differences mod $3$. Under
red+green $\to$ blue:

* $r-g \mapsto (r-1)-(g-1) = r-g$ — unchanged;
* $r-b \mapsto (r-1)-(b+2) = r-b-3 \equiv r-b \pmod 3$;
* $g-b \mapsto (g-1)-(b+2) = g-b-3 \equiv g-b \pmod 3$.

The other two kinds of meeting are the same computation with the colours permuted, so
**every pairwise difference of counts is conserved mod 3**. That is the wall.

**Read off the answer.** To finish monochromatic in colour $X$, the *other two* counts
must hit $0$ together, and $0-0 = 0$, so their difference must be $\equiv 0 \pmod 3$ —
and because that difference never moves mod $3$, it has to be $0$ **already** at the
start. Check the residues:

$$13 \equiv 1,\qquad 15 \equiv 0,\qquad 17 \equiv 2 \pmod 3.$$

The pairwise differences are $1-0 = 1$, $1-2 \equiv 2$, and $0-2 \equiv 1$ — **none**
is $0 \pmod 3$. No pair can vanish together, so the colony can **never** become one
colour. (Nudge a single chameleon, say $17 \to 18$ giving residues $1,0,0$, and now the
green–blue difference *is* $\equiv 0$, so all-red becomes reachable — the function
below decides any starting counts.)

**General criterion.** Colour $X$ is reachable as the final colour **iff the other two
counts are congruent mod 3**; the colony can be unified at all iff at least one of the
three pairs is congruent mod 3. Because congruence mod 3 is transitive, either no pair
matches — impossible, as here — or all three residues coincide and *every* colour is
reachable.

In [21]:
def chameleons_monochrome(a, b, c):
    """Can (a red, b green, c blue) chameleons all end one colour, under the rule
    'two of different colours meet -> both take the third colour'? A meeting does
    (-1, -1, +2) to the counts, leaving every pairwise difference unchanged mod 3.
    To finish all-X the other two colours must vanish together, so they must start
    congruent mod 3. Hence colour X is reachable iff the OTHER two counts agree
    mod 3. Returns (possible, sorted list of reachable final colours)."""
    if a + b + c == 0:
        return False, []
    other = {'red': (b, c), 'green': (a, c), 'blue': (a, b)}   # counts to zero out
    targets = sorted(col for col, (u, v) in other.items() if (u - v) % 3 == 0)
    return (len(targets) > 0), targets

for a, b, c in [(13, 15, 17), (13, 15, 18), (1, 1, 1), (13, 0, 0), (0, 3, 6)]:
    poss, tgt = chameleons_monochrome(a, b, c)
    print(f"(R{a:>2}, G{b:>2}, B{c:>2})  residues mod 3 = {(a % 3, b % 3, c % 3)}: "
          f"possible={poss}, final colours={tgt}")

(R13, G15, B17)  residues mod 3 = (1, 0, 2): possible=False, final colours=[]
(R13, G15, B18)  residues mod 3 = (1, 0, 0): possible=True, final colours=['red']
(R 1, G 1, B 1)  residues mod 3 = (1, 1, 1): possible=True, final colours=['blue', 'green', 'red']
(R13, G 0, B 0)  residues mod 3 = (1, 0, 0): possible=True, final colours=['red']
(R 0, G 3, B 6)  residues mod 3 = (0, 0, 0): possible=True, final colours=['blue', 'green', 'red']


---
## 2.8 Math Induction

> *To prove a statement $P(n)$ for all $n$, establish a small **base case** and then
> the **inductive step** — assume $P$ holds for the smaller instances and use that to
> force $P(n)$. Each case stands on the shoulders of the smaller ones, so a single
> local step propagates to infinity.*

**The template.** Two obligations:

1. **Base case.** Verify $P(n_0)$ directly for the smallest $n_0$.
2. **Inductive step.** Show $P(n-1)\Rightarrow P(n)$ (*weak* induction), or
   $\big(P(n_0)\wedge\cdots\wedge P(n-1)\big)\Rightarrow P(n)$ (*strong* induction,
   used when a problem breaks into two smaller pieces of unpredictable size).

The craft is choosing **what to induct on** and **where to peel**: the coin pile and
the chocolate bar each split into two smaller sub-problems (strong induction), while
the race track shrinks by *merging* two cans into one.

**Induction vs. the shortcut.** An induction proof certifies the result but can hide
*why* it holds and — for the race track — does not itself hand you the answer. The
chocolate bar and the race track each also have a slicker non-inductive route (a
piece-counting invariant and an $O(n)$ greedy scan), spelled out below. Where a
problem carries a size parameter it is solved for general $n$.

### 2.8.1 Coin split

**Problem.** Start with a pile of $n$ coins (book: $1000$). Split it into two non-empty
piles of sizes $a$ and $n-a$ and record the product $a(n-a)$. Keep splitting every pile
of size $>1$ the same way — recording the product of the two parts each time — until
only piles of size $1$ remain. Show the **sum of all recorded products** is always
$\dfrac{n(n-1)}{2}$, no matter how you split.

**Induction (strong, on the pile size).** Let $T(n)$ be the total for a pile of $n$;
the claim is $T(n)=\dbinom{n}{2}=\dfrac{n(n-1)}{2}$.

* **Base case.** $T(1)=0$: a lone coin is never split, and $\frac{1\cdot 0}{2}=0$. ✓
* **Inductive step.** Suppose the formula holds for **every** size $<n$. The first
  split cuts the pile into $a$ and $b=n-a$ (both $\ge 1$), contributing $ab$. After
  that the two sub-piles are split *independently* and both are smaller than $n$, so the
  hypothesis applies to each:
  $$T(n) \;=\; ab \;+\; T(a) \;+\; T(b) \;=\; ab \;+\; \frac{a(a-1)}{2} \;+\; \frac{b(b-1)}{2}.$$
  Using $a+b=n$,
  $$ab + \frac{a^2-a+b^2-b}{2} \;=\; ab + \frac{a^2+b^2-(a+b)}{2} \;=\; ab + \frac{a^2+b^2-n}{2},$$
  and since $a^2+b^2=(a+b)^2-2ab=n^2-2ab$,
  $$ab + \frac{n^2-2ab-n}{2} \;=\; \frac{n^2-n}{2} \;=\; \frac{n(n-1)}{2}.$$
  The split point $a$ **cancels**, so the total is identical for every splitting order. ∎

**Why $\binom{n}{2}$ (the intuition the induction confirms).** Treat the coins as $n$
distinct people. Splitting a pile into parts of size $a$ and $b$ separates exactly $ab$
*pairs* who were together and now never share a pile again — and each of the
$\binom{n}{2}$ pairs is separated at exactly one split (together at the start, apart in
the singletons at the end). Summing the products just counts every pair once.

In [22]:
import random

def coin_split_products(n, seed=None):
    """Split a pile of n coins repeatedly into two non-empty piles, each time
    recording (left size)*(right size), until every pile holds 1 coin. Returns the
    total of all recorded products for ONE random sequence of splits. Claim (proved
    by strong induction): the total is ALWAYS n(n-1)//2, whatever splits are chosen."""
    rng = random.Random(seed)
    piles, total = [n], 0
    while any(p > 1 for p in piles):
        i = rng.choice([j for j, p in enumerate(piles) if p > 1])   # a splittable pile
        p = piles.pop(i)
        a = rng.randint(1, p - 1)                                    # random split point
        total += a * (p - a)
        piles += [a, p - a]
    return total

for n in [5, 10, 100, 1000]:
    vals = {coin_split_products(n, seed=s) for s in range(40)}       # 40 random orders
    print(f"n={n:>4}: distinct totals over 40 random split orders = {vals}  "
          f"| n(n-1)/2 = {n * (n - 1) // 2}")

n=   5: distinct totals over 40 random split orders = {10}  | n(n-1)/2 = 10
n=  10: distinct totals over 40 random split orders = {45}  | n(n-1)/2 = 45
n= 100: distinct totals over 40 random split orders = {4950}  | n(n-1)/2 = 4950


n=1000: distinct totals over 40 random split orders = {499500}  | n(n-1)/2 = 499500


### 2.8.2 Chocolate bar

**Problem.** A chocolate bar is an $m\times n$ grid of $1\times 1$ squares (book:
$6\times 8 = 48$). A **break** takes one rectangular piece and snaps it in a straight
line along a groove into two smaller rectangles. How many breaks reduce the whole bar
to its $mn$ unit squares — and does the order matter?

**Induction (strong, on the number of squares).** Let $B(s)$ be the number of breaks to
fully split a piece of $s$ squares; the claim is $B(s)=s-1$.

* **Base case.** $B(1)=0$: a single square needs no breaks. ✓
* **Inductive step.** Assume the claim for all sizes $<s$. The first break splits the
  piece into two rectangles of $s_1$ and $s_2$ squares with $s_1+s_2=s$, both $<s$.
  Each is finished independently, so by the hypothesis
  $$B(s) = 1 + B(s_1) + B(s_2) = 1 + (s_1-1) + (s_2-1) = s_1+s_2-1 = s-1.$$
  Every strategy therefore uses exactly $s-1$ breaks — here $mn-1 = 47$. ∎

**Shortcut (a one-line invariant).** Drop the recursion and watch a single quantity:
**each break raises the number of pieces by exactly one** (one rectangle becomes two).
You start with $1$ piece and must finish with $mn$ pieces, so the number of breaks is
forced to be $mn-1$ — immediately, and manifestly independent of how or where you snap.
The induction above is just this counting argument unrolled.

### 2.8.3 Race track

**Problem.** On a one-way **circular** track sit $n$ fuel cans at various points; their
gas sums to **exactly one lap**. Your car starts with an **empty** tank, you may begin
at any can, and passing a can you collect its gas. Show you can always pick a starting
can from which you complete the full loop without ever running dry — and find one.

**Induction (on the number of cans) — existence.** Claim: whenever the total gas equals
one lap, a valid start exists.

* **Base case.** $n=1$: the lone can holds a full lap's worth of gas; start there. ✓
* **Inductive step.** Assume the claim for $n-1$ cans. With $n$ cans there **must be some
  can $i$ holding enough gas to reach the next can $i+1$**: if every can fell short of
  its gap to the next, the total gas would be strictly less than the total of the gaps —
  yet they are equal (one lap), a contradiction. Fix such a can $i$ and **merge $i+1$
  into $i$** — hand can $i$ the gas that was at $i+1$ and let it now span the stretch
  $i\to i+2$. Since can $i$ could already reach $i+1$, any journey that *arrives* at $i$
  behaves identically before and after the merge. This is a valid $(n-1)$-can instance
  whose total is still one lap, so by the hypothesis it has a valid start — and that same
  can is a valid start on the original $n$-can track. ∎

**Shortcut (an $O(n)$ greedy scan) — construction.** The induction shows a start
*exists* but not *which* one. Write $d_i=\text{gas}_i-\text{cost}_i$, the net fuel gained
crossing from can $i$ to $i+1$. Drive from can $0$ keeping a running tank $\sum d_i$;
**the moment the tank goes negative at can $i$, no can up to $i$ can be the start**, so
move the candidate start to $i+1$ and reset the tank to $0$. A single pass lands on a can
whose tank never dips below zero. It works because $\sum_i d_i = 0$: the cumulative fuel
returns to its start value after one lap, so the can **right after the global minimum** of
the running total keeps every partial sum non-negative — precisely the can the scan
settles on.

In [23]:
def race_track_start(gas, cost):
    """Circular track with n fuel cans: gas[i] litres sit at can i and cost[i] litres are
    needed to drive from can i to the next (indices mod n). The totals match
    (sum(gas) == sum(cost) == one lap). Starting with an empty tank at any can, return an
    index from which the whole loop completes with the tank never negative. O(n) greedy:
    whenever the running tank dips below 0, the start must lie after that point."""
    assert sum(gas) == sum(cost), "total gas must equal one lap"
    start, tank = 0, 0
    for i in range(len(gas)):
        tank += gas[i] - cost[i]
        if tank < 0:                        # cannot reach can i+1 from the current start
            start, tank = i + 1, 0
    return start % len(gas)

def _completes(gas, cost, start):
    """Simulate one lap from `start`; True iff the tank stays non-negative throughout."""
    tank, n = 0, len(gas)
    for k in range(n):
        i = (start + k) % n
        tank += gas[i] - cost[i]
        if tank < 0:
            return False
    return True

import random
random.seed(2)
allgood = True
for n in (5, 8, 20, 100):
    for _ in range(500):
        cost = [random.randint(1, 9) for _ in range(n)]
        gas = cost[:]; random.shuffle(gas)          # a permutation keeps sum(gas)==sum(cost)
        if not _completes(gas, cost, race_track_start(gas, cost)):
            allgood = False
    cost = [random.randint(1, 9) for _ in range(n)]
    gas = cost[:]; random.shuffle(gas)
    s = race_track_start(gas, cost)
    brute = [j for j in range(n) if _completes(gas, cost, j)]
    print(f"n={n:>3}: greedy start={s:>3}, completes loop={_completes(gas, cost, s)}, "
          f"#valid starts={len(brute)}, greedy is valid={s in brute}")
print("greedy start completes the loop on every one of 500 random instances per n:", allgood)

n=  5: greedy start=  1, completes loop=True, #valid starts=1, greedy is valid=True
n=  8: greedy start=  0, completes loop=True, #valid starts=1, greedy is valid=True
n= 20: greedy start=  9, completes loop=True, #valid starts=1, greedy is valid=True
n=100: greedy start= 20, completes loop=True, #valid starts=1, greedy is valid=True
greedy start completes the loop on every one of 500 random instances per n: True


---
## 2.9 Proof by Contradiction

> *To prove a statement, assume it is **false** and follow that assumption until it
> collides with something known to be true. The collision is impossible, so the
> assumption was wrong and the statement must hold.*

**The template.** Want to show "$P$ is true"? Suppose instead "$P$ is false," reason
carefully, and reach an absurdity — two things that cannot both hold. Since the only
soft spot in the chain was the initial assumption, $P$ must be true after all. It is
the natural tool for **"at least one …"** claims: assuming *none* of them happens is a
single, concrete thing to work with, and showing that leads nowhere proves at least
one must.

### 2.9.1 Rainbow hats

**Problem.** $n$ prisoners are each given a hat in one of $n$ "rainbow" colours (book:
$n = 7$). Colours may repeat — every prisoner could even get the same colour. Each
prisoner can see everyone else's hat but **not** their own. Then, **all at once** and
with no communication, each prisoner must say a single colour, guessing their own. They
may agree on a plan beforehand. Find a plan that **guarantees at least one correct
guess**, no matter how the hats are handed out.

**The one-sentence idea.** On their own, no prisoner can know their hat — from where
they stand it could be any of the $n$ colours, all equally possible, so a lone guess is
hopeless. The escape is teamwork: arrange for each prisoner to bet on a *different*
possibility for the whole picture, so that between them they cover **every** case — and
whoever bet on the true case is right.

**Step 1 — turn colours into numbers.** Label the $n$ colours $0, 1, \dots, n-1$. Write
$x_i$ for prisoner $i$'s own (unknown-to-them) colour. The quantity that secretly links
everyone is the grand total
$$S = x_0 + x_1 + \cdots + x_{n-1},$$
and in particular its remainder $S \bmod n$, which is some number in $\{0,1,\dots,n-1\}$.
Nobody knows this remainder, because nobody knows the full sum — each prisoner is missing
exactly one term, their own.

**Step 2 — hand out responsibilities.** Give prisoner $i$ the job of covering the single
case "the total leaves remainder $i$." Prisoner $i$ sees all the *other* hats, so they
know the sum of the other $n-1$ colours — call it $T_i$. They then announce the one colour
$g_i$ that would make the grand total land on remainder $i$:
$$g_i \;=\; (\,i - T_i\,) \bmod n .$$
(The logic: *if* my own hat were $g_i$, the total would be $T_i + g_i \equiv i \pmod n$.)
Each prisoner computes this from only what they can see, and the $n$ prisoners take the
$n$ different remainders $0,1,\dots,n-1$ — one each.

**Step 3 — exactly one bet is the truth.** The real total has some actual remainder, say
$S \equiv r \pmod n$. Look at prisoner $r$. Because $T_r = S - x_r$,
$$g_r = (r - T_r)\bmod n = (r - S + x_r)\bmod n = x_r ,$$
since $r - S \equiv 0 \pmod n$. So prisoner $r$'s guess is **exactly their own colour** —
a correct guess. Every prisoner covered a different remainder, and the truth matches one
of them, so (at least, and in fact exactly) one prisoner is right.

**Same fact, as a proof by contradiction.** Suppose the plan could fail — that on some
arrangement **every** prisoner guesses wrong, $g_i \ne x_i$ for all $i$. By construction,
prisoner $i$ is right *precisely* when $S \equiv i \pmod n$; so if prisoner $i$ is wrong,
then $S \not\equiv i \pmod n$. If **all** of them are wrong, then $S$ is congruent to none
of $0, 1, \dots, n-1$ — the total avoids every possible remainder. But dividing any
integer by $n$ must leave some remainder in $\{0,\dots,n-1\}$. Contradiction. So it is
impossible for all guesses to be wrong: **at least one is always correct.** $\;\blacksquare$

**Why it works for every $n$.** Nothing used the number $7$. The plan needs exactly one
prisoner per remainder, and there are $n$ remainders and $n$ prisoners — a perfect match —
so it guarantees a winner for any $n$.
Note: Modular figure n = no. of prisoners, not no. of colours

**Worked example — the book's case ($n = 7$ prisoners, $k = 7$ colours).** Here the number
of colours $k$ equals the number of prisoners $n = 7$, which is exactly what the covering
argument needs — one remainder $0,1,\dots,6$ for each prisoner. Suppose the hats are

$$x = (x_0, x_1, x_2, x_3, x_4, x_5, x_6) = (3,\,1,\,4,\,1,\,5,\,2,\,0).$$

The grand total is $S = 3+1+4+1+5+2+0 = 16$, so $S \bmod 7 = 2$. The general rule already
names the winner — **prisoner $2$** — before a single guess is computed. Now each prisoner
applies $g_i = (i - T_i)\bmod 7$ using the sum they actually see, $T_i = S - x_i = 16 - x_i$:

| prisoner $i$ | sees $T_i = 16 - x_i$ | guess $g_i = (i - T_i)\bmod 7$ | own $x_i$ | correct? |
|:---:|:---:|:---:|:---:|:---:|
| 0 | 13 | $(0-13)\bmod 7 = 1$ | 3 | ✗ |
| 1 | 15 | $(1-15)\bmod 7 = 0$ | 1 | ✗ |
| 2 | 12 | $(2-12)\bmod 7 = 4$ | 4 | ✓ |
| 3 | 15 | $(3-15)\bmod 7 = 2$ | 1 | ✗ |
| 4 | 11 | $(4-11)\bmod 7 = 0$ | 5 | ✗ |
| 5 | 14 | $(5-14)\bmod 7 = 5$ | 2 | ✗ |
| 6 | 16 | $(6-16)\bmod 7 = 4$ | 0 | ✗ |

Exactly one hit — prisoner $2$, precisely the index singled out by $S \bmod 7 = 2$. That is
the general identity of Step 3 in numbers: for the winning index $r = S \bmod n = 2$,
$$g_r = (r - T_r)\bmod n = (r - S + x_r)\bmod n = x_r \quad\text{because } r - S \equiv 0 \pmod n .$$
Every other prisoner bet on a total that was not the real one, so they miss — which is
fine, since the team needs only a single correct voice. (These are exactly the guesses
`[1, 0, 4, 2, 0, 5, 4]` the code prints below.)

The function below runs the strategy for any hat assignment and confirms that in every
random game exactly one prisoner guesses right.

In [24]:
import random

def rainbow_hats_guesses(hats):
    """n prisoners each wear a hat coloured 0..n-1 (colours may repeat). Prisoner i sees
    every hat except their own and simultaneously guesses their own colour. Strategy:
    prisoner i assumes the grand total of all n colours leaves remainder i (mod n) and
    names the colour that makes it so, g_i = (i - sum of the OTHER hats) mod n. The
    prisoner whose index equals the true total mod n is always right, so exactly one
    guesses correctly. Returns the guesses, each computed only from what that prisoner
    can see."""
    n = len(hats)
    total = sum(hats)
    return [(i - (total - hats[i])) % n for i in range(n)]   # total - hats[i] = what i sees

random.seed(0)
for n in (2, 3, 7, 26, 100):
    per_game = {sum(g == h for g, h in zip(rainbow_hats_guesses(hats), hats))
                for hats in ([random.randrange(n) for _ in range(n)] for _ in range(20000))}
    print(f"n={n:>3}: #correct guesses per game over 20000 games = {sorted(per_game)}  "
          f"(always exactly one)")

hats = [3, 1, 4, 1, 5, 2, 0]                       # a concrete 7-prisoner game
guesses = rainbow_hats_guesses(hats)
print("\nexample (n=7): hats    =", hats, " total mod 7 =", sum(hats) % 7)
print("               guesses =", guesses,
      "-> correct prisoner(s):", [i for i in range(7) if guesses[i] == hats[i]])

n=  2: #correct guesses per game over 20000 games = [1]  (always exactly one)
n=  3: #correct guesses per game over 20000 games = [1]  (always exactly one)


n=  7: #correct guesses per game over 20000 games = [1]  (always exactly one)


n= 26: #correct guesses per game over 20000 games = [1]  (always exactly one)


n=100: #correct guesses per game over 20000 games = [1]  (always exactly one)

example (n=7): hats    = [3, 1, 4, 1, 5, 2, 0]  total mod 7 = 2
               guesses = [1, 0, 4, 2, 0, 5, 4] -> correct prisoner(s): [2]


---
*That rounds out Chapter 2's brain-teaser techniques (§2.1–§2.9). Onward to Chapter 3.*

---
# Chapter 3 · Calculus and Linear Algebra

> Switching gears from brain teasers to the **calculus toolkit**. Since I'm writing
> these from memory rather than the textbook, the next two sections are deliberately
> **comprehensive reference notes** — the differentiation and integration rules, the
> standard tables, and the applications that actually surface in quant interviews
> (Taylor expansions, optimisation, expectations of continuous random variables).
> This material is formula-centric, so it is all **markdown, no code**.

## 3.1 Differentiation

### The derivative
A function $f$ is **differentiable** at $x$ if the limit
$$f'(x) \;=\; \frac{df}{dx} \;=\; \lim_{h\to 0}\frac{f(x+h)-f(x)}{h}$$
exists. Geometrically $f'(x)$ is the **slope of the tangent** at $x$; physically it is an
instantaneous rate of change. Differentiability at a point implies continuity there — but
not conversely ($f(x)=\lvert x\rvert$ is continuous yet has no derivative at $0$).

**Limit facts that power the rules.**
$$\lim_{x\to 0}\frac{\sin x}{x}=1,\qquad \lim_{x\to 0}\frac{1-\cos x}{x}=0,\qquad
\lim_{x\to 0}\frac{e^{x}-1}{x}=1,\qquad \lim_{n\to\infty}\Big(1+\tfrac{x}{n}\Big)^{n}=e^{x}.$$

### The differentiation rules
For differentiable $u,v$ and constants $a,b,c$:

| rule | statement |
|------|-----------|
| **Linearity** | $(a\,u+b\,v)' = a\,u' + b\,v'$ |
| **Product** | $(uv)' = u'v + uv'$ |
| **Quotient** | $\left(\dfrac{u}{v}\right)' = \dfrac{u'v - uv'}{v^{2}}$ |
| **Chain** | $\dfrac{d}{dx}\,f(g(x)) = f'(g(x))\,g'(x)$ |
| **Reciprocal** | $\left(\dfrac{1}{v}\right)' = -\dfrac{v'}{v^{2}}$ |
| **Inverse function** | $\left(f^{-1}\right)'(y) = \dfrac{1}{f'(x)}$ at $y=f(x)$ |
| **General power** | $\dfrac{d}{dx}\,x^{r} = r\,x^{r-1}$ for any real $r$ |

**Implicit differentiation.** If $x,y$ satisfy $F(x,y)=0$, differentiate both sides in $x$
treating $y=y(x)$ (chain rule) and solve for $y'$. E.g. $x^{2}+y^{2}=1 \Rightarrow
2x+2y\,y'=0 \Rightarrow y'=-x/y$.

**Logarithmic differentiation.** For products, powers, or variable exponents, take $\ln$
first. For $y=f(x)^{g(x)}$,
$$\ln y = g\ln f \;\Rightarrow\; \frac{y'}{y}=g'\ln f + g\,\frac{f'}{f}
\;\Rightarrow\; y' = f^{g}\!\left(g'\ln f + \frac{g\,f'}{f}\right),$$
which gives the classic $\dfrac{d}{dx}\,x^{x} = x^{x}(\ln x + 1)$.

### Standard derivatives

| $f(x)$ | $f'(x)$ | &nbsp; | $f(x)$ | $f'(x)$ |
|---|---|---|---|---|
| $c$ (constant) | $0$ | | $\sin x$ | $\cos x$ |
| $x^{r}$ | $r\,x^{r-1}$ | | $\cos x$ | $-\sin x$ |
| $e^{x}$ | $e^{x}$ | | $\tan x$ | $\sec^{2}x$ |
| $a^{x}$ | $a^{x}\ln a$ | | $\sec x$ | $\sec x\tan x$ |
| $\ln x$ | $1/x$ | | $\cot x$ | $-\csc^{2}x$ |
| $\log_a x$ | $1/(x\ln a)$ | | $\csc x$ | $-\csc x\cot x$ |
| $\arcsin x$ | $1/\sqrt{1-x^{2}}$ | | $\sinh x$ | $\cosh x$ |
| $\arccos x$ | $-1/\sqrt{1-x^{2}}$ | | $\cosh x$ | $\sinh x$ |
| $\arctan x$ | $1/(1+x^{2})$ | | $\tanh x$ | $\operatorname{sech}^{2}x$ |

**Higher-order derivatives** iterate: $f''=(f')'$, and the $n$-th is $f^{(n)}$. Two
patterns worth memorising: $\dfrac{d^{n}}{dx^{n}}e^{ax}=a^{n}e^{ax}$, and the **Leibniz
rule** $(uv)^{(n)}=\displaystyle\sum_{k=0}^{n}\binom{n}{k}u^{(k)}v^{(n-k)}$.

### Applications of differentiation

**Tangent line & linear approximation.** Near $x=a$, $f(x)\approx f(a)+f'(a)(x-a)$ — the
first-order approximation behind every "sensitivity" or *delta* calculation.

**Extrema.** Interior extrema sit at **critical points** ($f'=0$ or $f'$ undefined).
- *First-derivative test:* $f'$ flips $+\!\to\!-$ ⇒ local max; $-\!\to\!+$ ⇒ local min.
- *Second-derivative test:* at $f'(c)=0$, $f''(c)<0$ ⇒ max, $f''(c)>0$ ⇒ min, $f''(c)=0$ ⇒
  inconclusive.
- *Global* extrema on $[a,b]$: compare critical points **and** the endpoints.

**Concavity & inflection.** $f''>0$ ⇒ convex (concave up); $f''<0$ ⇒ concave down; a sign
change of $f''$ marks an inflection point.

**Mean Value Theorem.** If $f$ is continuous on $[a,b]$ and differentiable on $(a,b)$, some
$c$ satisfies $f'(c)=\dfrac{f(b)-f(a)}{b-a}$. (Rolle's theorem is the case $f(a)=f(b)$.)

**L'Hôpital's rule.** For $\tfrac{0}{0}$ or $\tfrac{\infty}{\infty}$ forms,
$\displaystyle\lim\frac{f}{g}=\lim\frac{f'}{g'}$ when the right side exists. Other
indeterminate forms ($0\cdot\infty$, $\infty-\infty$, $1^{\infty}$, $0^{0}$,
$\infty^{0}$) are first rearranged into a quotient, often by taking $\ln$.

**Taylor / Maclaurin series.** Near $x=a$,
$$f(x)=\sum_{n=0}^{\infty}\frac{f^{(n)}(a)}{n!}(x-a)^{n}
=f(a)+f'(a)(x-a)+\frac{f''(a)}{2!}(x-a)^{2}+\cdots$$
Core expansions about $0$:
$$e^{x}=\sum_{n\ge0}\frac{x^{n}}{n!},\quad
\sin x=x-\frac{x^{3}}{3!}+\cdots,\quad \cos x=1-\frac{x^{2}}{2!}+\cdots,$$
$$\ln(1+x)=x-\frac{x^{2}}{2}+\frac{x^{3}}{3}-\cdots\ (|x|<1),\quad
(1+x)^{\alpha}=1+\alpha x+\frac{\alpha(\alpha-1)}{2!}x^{2}+\cdots$$
In finance the **second-order** expansion is everywhere: a value $V(S)$ moves by
$\Delta V\approx V'(S)\,\Delta S+\tfrac12 V''(S)\,(\Delta S)^{2}$ — precisely **delta +
gamma**, the first two option Greeks.

**Newton's method.** To solve $f(x)=0$, iterate $x_{n+1}=x_{n}-\dfrac{f(x_{n})}{f'(x_{n})}$;
convergence is quadratic near a simple root.

**Related rates.** Differentiate a relation in time $t$ to link rates — e.g. from
$V=\tfrac43\pi r^{3}$, $\dfrac{dV}{dt}=4\pi r^{2}\dfrac{dr}{dt}$.

## 3.2 Integration

### Antiderivatives and the definite integral
$F$ is an **antiderivative** of $f$ when $F'=f$; the **indefinite integral** is
$\int f\,dx=F(x)+C$. The **definite integral** $\int_a^b f\,dx$ is the signed area under
$f$, defined as a limit of Riemann sums.

**Fundamental Theorem of Calculus.**
1. If $F(x)=\displaystyle\int_a^x f(t)\,dt$ then $F'(x)=f(x)$ — differentiation undoes
   integration.
2. $\displaystyle\int_a^b f(x)\,dx=F(b)-F(a)$ for any antiderivative $F$.
- *Leibniz rule (variable limits):*
  $\dfrac{d}{dx}\displaystyle\int_{a(x)}^{b(x)}f(t)\,dt=f(b(x))\,b'(x)-f(a(x))\,a'(x)$.

**Basic properties.** $\int(af+bg)=a\int f+b\int g$; $\int_a^b=-\int_b^a$;
$\int_a^b=\int_a^c+\int_c^b$; and if $f\ge 0$ then $\int_a^b f\ge 0$.

### Standard antiderivatives (add $+\,C$)

| $f(x)$ | $\int f\,dx$ | &nbsp; | $f(x)$ | $\int f\,dx$ |
|---|---|---|---|---|
| $x^{r}\ (r\ne -1)$ | $\dfrac{x^{r+1}}{r+1}$ | | $\sin x$ | $-\cos x$ |
| $1/x$ | $\ln\lvert x\rvert$ | | $\cos x$ | $\sin x$ |
| $e^{x}$ | $e^{x}$ | | $\sec^{2}x$ | $\tan x$ |
| $a^{x}$ | $a^{x}/\ln a$ | | $\dfrac{1}{1+x^{2}}$ | $\arctan x$ |
| $\ln x$ | $x\ln x-x$ | | $\dfrac{1}{\sqrt{1-x^{2}}}$ | $\arcsin x$ |

### Techniques of integration

**Substitution ($u$-sub)** — reverses the chain rule. With $u=g(x)$, $du=g'(x)\,dx$,
$$\int f(g(x))\,g'(x)\,dx=\int f(u)\,du.$$

**Integration by parts** — reverses the product rule,
$$\int u\,dv=uv-\int v\,du.$$
Choose $u$ by **LIATE** (Log, Inverse-trig, Algebraic, Trig, Exponential — earlier in the
list ⇒ take as $u$). E.g. $\int x e^{x}\,dx=xe^{x}-e^{x}+C$ and
$\int \ln x\,dx=x\ln x-x+C$.

**Partial fractions** — for a rational $\tfrac{P(x)}{Q(x)}$ with $\deg P<\deg Q$, split
into simpler pieces before integrating, e.g.
$$\frac{1}{(x-a)(x-b)}=\frac{1}{a-b}\left(\frac{1}{x-a}-\frac{1}{x-b}\right),$$
which integrates to logarithms (and, for irreducible quadratics, arctangents).

**Trigonometric substitution** — for $\sqrt{a^{2}-x^{2}}$ set $x=a\sin\theta$; for
$\sqrt{a^{2}+x^{2}}$ set $x=a\tan\theta$; for $\sqrt{x^{2}-a^{2}}$ set $x=a\sec\theta$.

**Improper integrals** — infinite limits or unbounded integrands are defined by a limit,
$\int_a^\infty f=\lim_{b\to\infty}\int_a^b f$. Benchmarks: $\int_1^\infty x^{-p}\,dx$
converges **iff** $p>1$, while $\int_0^1 x^{-p}\,dx$ converges **iff** $p<1$.

### Applications of integration

**Geometry.**
- *Area between curves:* $\displaystyle\int_a^b\big(f(x)-g(x)\big)\,dx$.
- *Volume of revolution:* discs $\displaystyle\int_a^b\pi\,[f(x)]^{2}\,dx$; shells
  $\displaystyle\int_a^b 2\pi x\,f(x)\,dx$.
- *Arc length:* $\displaystyle\int_a^b\sqrt{1+[f'(x)]^{2}}\,dx$.
- *Average value:* $\bar f=\dfrac{1}{b-a}\displaystyle\int_a^b f\,dx$.

**Probability — the quant workhorse.** For a continuous random variable with density $f$:
- it integrates to one, $\displaystyle\int_{-\infty}^{\infty}f(x)\,dx=1$;
- the CDF is $F(x)=\displaystyle\int_{-\infty}^{x}f$, and $P(a\le X\le b)=\int_a^b f$;
- **expectation** $\mathbb{E}[g(X)]=\displaystyle\int g(x)f(x)\,dx$, so
  $\mathbb{E}[X]=\int x f\,dx$ and $\operatorname{Var}(X)=\mathbb{E}[X^{2}]-\mathbb{E}[X]^{2}$;
- **tail formula:** for $X\ge 0$, $\mathbb{E}[X]=\displaystyle\int_0^\infty P(X>x)\,dx$.

**Two integrals to know cold.**
- *Gaussian:* $\displaystyle\int_{-\infty}^{\infty}e^{-x^{2}}\,dx=\sqrt{\pi}$, which is why
  the $\mathcal{N}(\mu,\sigma^{2})$ density
  $\tfrac{1}{\sigma\sqrt{2\pi}}\,e^{-(x-\mu)^{2}/2\sigma^{2}}$ integrates to $1$.
- *Gamma:* $\displaystyle\int_0^\infty x^{n}e^{-x}\,dx=n!$ (generally
  $\Gamma(t)=\int_0^\infty x^{t-1}e^{-x}\,dx$, with $\Gamma(n)=(n-1)!$).

**Discounting (finance).** The present value of a continuous cash-flow $c(t)$ at continuous
rate $r$ is $\displaystyle\int_0^T c(t)\,e^{-rt}\,dt$; a perpetual constant flow $c$ gives
$\displaystyle\int_0^\infty c\,e^{-rt}\,dt=\dfrac{c}{r}$.

---
*More of Chapter 3 as I keep reading: partial derivatives & multiple integrals, ODEs, and linear algebra.*